# vjaudit pipeline notebook

This notebook runs the whole pipeline, from the dataset files to the final results table: load
questions, check videos, draw samples, extract frames, let video models answer, let judge models grade
the answers, export labelling batches for Claude, import the labels, and report.

Everything happens here, in plain Python cells you can read and change. The notebook does not import
the `vjaudit` package; it only reads the prompt texts from `../configs/prompts/`.

<a id="toc"></a>
**Table of contents**

- [0. How to use this notebook](#s0)
- [1. Prerequisites: install and download](#s1)
- [2. Settings](#s2)
- [3. Helpers](#s3)
- [4. Load the benchmarks](#s4)
- [5. Check the videos](#s5)
- [6. Draw the samples](#s6)
- [7. Frames](#s7)
- [8. Generate answers](#s8)
- [9. Judge the answers](#s9)
- [10. Label with Claude](#s10)
- [11. Report](#s11)
- [12. Troubleshooting and next steps](#s12)

<a id="s0"></a>
## 0. How to use this notebook

**The idea in two sentences.** A *judge* model grades free-text answers of video models against the
benchmark's reference answer. This project measures how much those grades can be trusted, and whether
the judge treats some models (for example ones answering *without* the video) more kindly than others.
`../docs/EXPLAINED.md` explains it at length.

**Running it.**

1. Open the notebook in JupyterLab or VS Code with the `dl` environment as the kernel.
2. Set your choices in section 2 (normally only `PROFILE` and `RUN`).
3. Run the cells from top to bottom (JupyterLab: *Run → Run All Cells*), or one section at a time.

**Mac or server.** One switch in section 2, `PROFILE = "mac"` or `"server"`, chooses the models, judges,
sample sizes and judge stages. Every section ends its explanation with a **Mac vs server** line that
says what (if anything) differs.

**Two runs.**

| Run | Setting | Questions per benchmark (Mac) | What happens |
|---|---|---|---|
| Run 1 | `RUN = "annotated"` | 100 | answers, judging, Claude labels, report |
| Run 2 | `RUN = "main"` | 1,000 | more answers and judging with the main judge; Run 1's work is reused |

**Stopping and resuming.** Every long step saves its results as it goes. If the kernel stops, the laptop
sleeps or you press *Stop*, run the same section again: finished work is skipped. After a kernel
restart, run sections 2 and 3 first (settings and helpers), then jump to any section; each section reloads
what it needs from disk and tells you which earlier section to run if something is missing.

**Try it first in a dry run.** With `DRY_RUN = True` the notebook uses 3 questions per benchmark and
stand-in models that load nothing. It runs end to end in about a minute and checks that your data and
folders are set up. Its results go to a separate folder.

**Where results go.** Everything is written to `notebooks/runs/<profile>/` (git ignores it):

| File or folder | Contents |
|---|---|
| `pool.jsonl` | every question of the three benchmarks, with its video file and whether it decodes |
| `items.jsonl` | the sampled questions and the sets they belong to |
| `generations.jsonl` | every answer of every system |
| `judgecalls.jsonl` | every judge verdict, margin and raw reply |
| `units.jsonl`, `labels.jsonl` | the distinct answers given to Claude; the imported labels |
| `frames/`, `judge_cache/` | cached video frames and judge replies |
| `annotation/claude/` | batch files for Claude, and a `replies/` folder for Claude's answers |
| `results.md` | the final report |

**Long runs on a Mac.** Keep the laptop plugged in. To stop it from sleeping, run `caffeinate -dimsu` in a
terminal while the notebook works.

<a id="s1"></a>
## 1. Prerequisites: install and download

Do these once. They are **commented out** so that *Run All* never reinstalls packages or starts a
130 GB download by accident. To run one, remove the `#` in front of the command and run the cell, or
paste the command (without the leading `!`) into a terminal in the `vjauditv2` folder.

**1a. Packages.** Install them into the environment this notebook's kernel uses (`dl`):

- on the Mac: `torch`, `transformers` and helpers (`mac` extra), plus Jupyter, progress bars and plots
  (`notebook` extra);
- on the GPU server: `vllm` instead (`gpu` extra);
- the OpenAI client (`openai`) is already part of the base install; it is only needed for the optional
  GPT-3.5 stages.

In [ ]:
# Mac (run once, in the dl environment):
# !pip install -e "..[mac,notebook]"
#
# GPU server:
# !pip install -e "..[gpu,notebook]"

**1b. Datasets.** The download scripts fetch the three benchmarks (questions and videos) into
`vjauditv2/data/` by default. You already downloaded them into `vjaudit/data/`, and section 2 points
there on the Mac, so **skip this on your Mac**. On a new machine, run one of these:

| Script | What it fetches | Size |
|---|---|---|
| `download_all.sh` | all three benchmarks, and creates `configs/local/paths.yaml` | about 135 GB |
| `download_msvd.sh` | MSVD-QA questions (MEGA) and the 1,970 YouTube clips | about 3.5 GB |
| `download_anetqa.sh` | ActivityNet-QA questions (GitHub) and 5,044 videos (Hugging Face) | about 125 GB |
| `download_nextqa.sh` | NExT-QA open-ended and multiple-choice questions, and 1,570 videos | about 6 GB |

Each script accepts `--annotations-only` to fetch only the question files (a few MB), which is enough to
look at the data in section 4 before the videos arrive. MSVD needs `megatools`
(`brew install megatools` on a Mac).

In [ ]:
# Everything (questions + videos), into ../data:
# !bash ../scripts/download/download_all.sh
#
# Or one benchmark at a time:
# !bash ../scripts/download/download_msvd.sh
# !bash ../scripts/download/download_anetqa.sh
# !bash ../scripts/download/download_nextqa.sh
#
# Question files only (fast), for a first look:
# !bash ../scripts/download/download_all.sh --annotations-only

**1c. Judge prompt check (optional, once).** The T-VCG judge prompt should be word for word the
official Video-ChatGPT prompt. This script fetches the official one and compares. Without it, the
notebook still runs; the report then notes that the prompts were not verified.

In [ ]:
# !bash ../scripts/download/fetch_upstream_prompts.sh

<a id="s2"></a>
## 2. Settings

**The next cell is the only one you normally edit.** Run it, and the cell after it, every time you
start the notebook.

| Setting | Mac | Server | Meaning |
|---|---|---|---|
| `PROFILE` | `"mac"` | `"server"` | which models, judges, sample sizes and stages to use |
| `DRY_RUN` | `False` (`True` for a 1-minute check) | same | 3 questions per benchmark, stand-in models, separate results folder |
| `DATA_ROOT` | `None` → `../../vjaudit/data` | `None` → `../data` | folder holding `msvd/`, `anetqa/`, `nextqa/` |
| `RUN` | `"annotated"` (Run 1), later `"main"` (Run 2) | same | which sample the answer and judge steps work on |
| `RUN_LIMIT` | `None`, or `5` for a first real test | same | at most this many questions per benchmark in sections 8–9 |
| `RUN_STAGES` | `None` | `None` | judge stages to run; `None` = all non-optional stages of this run |
| `REBUILD_POOL` / `REBUILD_SAMPLE` | `False` | `False` | `True` redoes section 5 / 6 from scratch (old tables are kept with a timestamp) |

**Mac vs server:** change `PROFILE`. On the server, also make sure `OPENAI_API_KEY` is set in the
environment before starting Jupyter (the GPT-3.5 judge needs it).

In [ ]:
# ── The settings you normally change ─────────────────────────────────────────────────────────────
PROFILE = "mac"          # "mac" on the MacBook · "server" on the GPU server
DRY_RUN = False          # True = 3 questions per benchmark + stand-in models, to check the notebook in ~1 minute
DATA_ROOT = None         # None = Mac: ../../vjaudit/data (v1's data folder) · server: ../data · or a path string
RUN = "annotated"        # "annotated" = Run 1 · "main" = Run 2 (reuses everything Run 1 did)
RUN_LIMIT = None         # None = all questions · 5 = a quick first real test (5 questions per benchmark)
RUN_STAGES = None        # None = every non-optional judge stage of this run · e.g. ["M1", "M2"] or ["G1", "G2"]
REBUILD_POOL = False     # True = redo section 5 (video check) from scratch
REBUILD_SAMPLE = False   # True = redraw the samples in section 6 (do this only before generating answers)

The next cell loads the Python libraries the notebook uses. `tqdm` (progress bars) is optional.

In [ ]:
import ast
import copy
import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import string
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:  # progress bars are optional
    def tqdm(iterable=None, **kwargs):
        return iterable

pd.set_option("display.max_colwidth", 80)

try:
    display  # provided by Jupyter
except NameError:  # plain Python (tests): print instead
    def display(obj):
        print(obj)

The next cell lists the models, judges and judge stages of each profile. Edit it only to change the set-up.

- **Generators** (video models). `styles` adds one system per answer style (`short` = one word or phrase,
  `sentence` = one sentence plus a reason). `blind: True` adds a system that answers *without* the video.
  `frame_ablation` adds systems that see only k frames. `revision: None` uses the latest version on the
  Hugging Face Hub; put a commit id there to pin it (section 8 prints the ids it used).
- **Judges.** `transformers` judges run locally; `openai` judges call the API (only the GPT-3.5 entries).
- **Stages.** One block of judging: which set, which judge, which prompt template, and the decoding.
  `optional: True` stages run only when named in `RUN_STAGES`.

**Mac vs server:** nothing to edit here; `PROFILE` picks `MAC` or `SERVER`.

In [ ]:
T0_VCG = dict(temperature=0, max_tokens=64, logprobs=True, top_logprobs=20, n=1)  # greedy, T-VCG reply
T0_RUB = dict(temperature=0, max_tokens=1, logprobs=True, top_logprobs=20, n=1)   # greedy, one-word reply


def _stage(name, judge, subset, template, request, systems="core", decoding="t0", optional=False):
    return dict(name=name, judge_id=judge, subset=subset, systems=systems, template=template,
                decoding_tag=decoding, request=request, optional=optional)


GPT35 = [
    dict(id="gpt-3.5-turbo-0125", backend="openai", model="gpt-3.5-turbo-0125", max_usd=100,
         price_in_per_mtok=0.50, price_out_per_mtok=1.50),
    dict(id="gpt-3.5-turbo-1106", backend="openai", model="gpt-3.5-turbo-1106", max_usd=20,
         price_in_per_mtok=0.50, price_out_per_mtok=1.50),
]

MAC = dict(
    device="mps",
    generators=[
        dict(id="qwen25vl3b", family="qwen-vl", repo="Qwen/Qwen2.5-VL-3B-Instruct", revision=None,
             backend="transformers", dtype="auto", n_frames={"default": 8}, styles=["short", "sentence"],
             blind=True, frame_ablation=[]),
        dict(id="qwen3vl2b", family="qwen-vl", repo="Qwen/Qwen3-VL-2B-Instruct", revision=None,
             backend="transformers", dtype="auto", n_frames={"default": 8}, styles=["short"],
             blind=False, frame_ablation=[]),
        dict(id="llavaov05b", family="llava-onevision", repo="llava-hf/llava-onevision-qwen2-0.5b-ov-hf",
             revision=None, backend="transformers", dtype="auto", n_frames={"default": 8}, styles=["short"],
             blind=False, frame_ablation=[]),
    ],
    frame_max_side=448,
    judges=[
        dict(id="qwen3-4b", backend="transformers", model="Qwen/Qwen3-4B-Instruct-2507", revision=None),
        dict(id="qwen25-1.5b", backend="transformers", model="Qwen/Qwen2.5-1.5B-Instruct", revision=None),
        GPT35[0],
    ],
    samples=dict(main=1000, annotated=100, vid=30),
    stages=[
        _stage("M1", "qwen3-4b", "annotated", "t-vcg", T0_VCG),
        _stage("M2", "qwen3-4b", "annotated", "t-rub", T0_RUB),
        _stage("M3", "qwen25-1.5b", "annotated", "t-vcg", T0_VCG),
        _stage("M4", "qwen25-1.5b", "annotated", "t-rub", T0_RUB),
        _stage("M5", "qwen3-4b", "main", "t-vcg", T0_VCG),
        _stage("M6", "qwen3-4b", "main", "t-rub", T0_RUB),
        _stage("G1", "gpt-3.5-turbo-0125", "annotated", "t-vcg", T0_VCG, optional=True),
        _stage("G2", "gpt-3.5-turbo-0125", "annotated", "t-rub", T0_RUB, optional=True),
    ],
)

_SERVER_FRAMES = {"default": 16, "anetqa": 32}
SERVER = dict(
    device="cuda",
    generators=[
        dict(id="qwen25vl7b", family="qwen-vl", repo="Qwen/Qwen2.5-VL-7B-Instruct", revision=None, backend="vllm",
             dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=32768, styles=["short"], blind=True,
             frame_ablation=[]),
        dict(id="qwen3vl8b", family="qwen-vl", repo="Qwen/Qwen3-VL-8B-Instruct", revision=None, backend="vllm",
             dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=32768, styles=["short", "sentence"],
             blind=True, frame_ablation=[1, 4]),
        dict(id="qwen3vl2b", family="qwen-vl", repo="Qwen/Qwen3-VL-2B-Instruct", revision=None, backend="vllm",
             dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=32768, styles=["short"], blind=False,
             frame_ablation=[]),
        dict(id="internvl35_8b", family="internvl", repo="OpenGVLab/InternVL3_5-8B", revision=None, backend="vllm",
             dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=32768, styles=["short"], blind=True,
             frame_ablation=[]),
        dict(id="llavanextvideo7b", family="llava-next-video", repo="llava-hf/LLaVA-NeXT-Video-7B-hf", revision=None,
             backend="vllm", dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=8192, styles=["short"],
             blind=False, frame_ablation=[]),
    ],
    frame_max_side=768,
    judges=GPT35,
    samples=dict(main=2000, annotated=300, vid=200),
    stages=[
        _stage("S1", "gpt-3.5-turbo-0125", "annotated", "t-vcg", T0_VCG),
        _stage("S2", "gpt-3.5-turbo-0125", "annotated", "t-rub", T0_RUB),
        _stage("S3", "gpt-3.5-turbo-0125", "main", "t-vcg", T0_VCG),
        _stage("S4", "gpt-3.5-turbo-0125", "main", "t-rub", T0_RUB),
        _stage("S5", "gpt-3.5-turbo-0125", "annotated", "t-vcg",
               dict(temperature=None, max_tokens=64, logprobs=True, top_logprobs=20, n=3), decoding="default-n3"),
        _stage("S6a", "gpt-3.5-turbo-0125", "annotated", "t-vcg", T0_VCG, systems="ablation"),
        _stage("S6b", "gpt-3.5-turbo-0125", "annotated", "t-rub", T0_RUB, systems="ablation"),
        _stage("S6c", "gpt-3.5-turbo-0125", "main", "t-vcg", T0_VCG, systems="ablation"),
        _stage("S6d", "gpt-3.5-turbo-0125", "main", "t-rub", T0_RUB, systems="ablation"),
        _stage("S7", "gpt-3.5-turbo-1106", "annotated", "t-vcg", T0_VCG),
    ],
)

SHARED = dict(seed=42, msvd_min_per=15, frac_dev=0.4, max_new_tokens={"short": 128, "sentence": 256, "mc": 8})
BENCHMARKS = ["msvd", "anetqa", "nextoe"]


def resolve_settings(params: dict, repo: Path) -> dict:
    """Combine the switches above with the chosen profile into one settings dict ``S``."""
    profile = params["PROFILE"]
    if profile not in ("mac", "server"):
        raise ValueError(f'PROFILE must be "mac" or "server", not {profile!r}')
    if params.get("RUN", "annotated") not in ("annotated", "main"):
        raise ValueError('RUN must be "annotated" or "main"')
    base = copy.deepcopy(params.get("MAC", MAC) if profile == "mac" else params.get("SERVER", SERVER))
    dry = bool(params.get("DRY_RUN"))
    run = params.get("RUN", "annotated")
    if params.get("DATA_ROOT"):
        data_root = Path(params["DATA_ROOT"]).expanduser()
    else:
        data_root = repo.parent / "vjaudit" / "data" if profile == "mac" else repo / "data"
    if dry:
        base["samples"] = dict(main=3, annotated=3, vid=1)
        for g in base["generators"]:
            g["backend"] = "fake"
        for j in base["judges"]:
            j["backend"] = "fake"
    wanted = params.get("RUN_STAGES")
    if wanted:
        unknown = sorted(set(wanted) - {st["name"] for st in base["stages"]})
        if unknown:
            raise ValueError(f"unknown stage(s) {unknown}; stages are {[st['name'] for st in base['stages']]}")
        stages = [st for st in base["stages"] if st["name"] in wanted]
    else:
        stages = [st for st in base["stages"] if st["subset"] == run and not st["optional"]]
    return dict(SHARED, profile=profile, dry_run=dry, run=run, run_limit=params.get("RUN_LIMIT"),
                data_root=data_root, repo=repo, prompts=repo / "configs" / "prompts",
                runs=repo / "notebooks" / "runs" / (profile + ("-dryrun" if dry else "")),
                device=base["device"], generators=base["generators"], judges=base["judges"],
                frame_max_side=base["frame_max_side"], samples=base["samples"], stages=stages,
                all_stages=base["stages"], rebuild_pool=bool(params.get("REBUILD_POOL")),
                rebuild_sample=bool(params.get("REBUILD_SAMPLE")))

def detect_device(preferred: str) -> str:
    """The device that will really be used: the preferred one if torch can use it, else cpu."""
    try:
        import torch
    except ImportError:
        return "cpu (torch not installed)"
    if preferred == "mps" and torch.backends.mps.is_available():
        return "mps"
    if preferred == "cuda" and torch.cuda.is_available():
        return "cuda"
    return "cpu"


def show_settings(S: dict) -> None:
    """Print the resolved settings, the device and the main package versions."""
    rows = [("profile", S["profile"] + ("  (DRY RUN)" if S["dry_run"] else "")), ("run", S["run"]),
            ("run_limit", S["run_limit"]), ("data_root", S["data_root"]),
            ("data_root exists", S["data_root"].exists()), ("results folder", S["runs"]),
            ("samples (main/annotated/vid)", "/".join(str(S["samples"][k]) for k in ("main", "annotated", "vid"))),
            ("generators", ", ".join(f'{g["id"]} [{g["backend"]}]' for g in S["generators"])),
            ("judges", ", ".join(f'{j["id"]} [{j["backend"]}]' for j in S["judges"])),
            ("judge stages this run", ", ".join(st["name"] for st in S["stages"]) or "none"),
            ("frames", f'{S["generators"][0]["n_frames"]} at max {S["frame_max_side"]} px'),
            ("device", "none needed (dry run)" if S["dry_run"] else detect_device(S["device"]))]
    for mod in ("torch", "transformers", "vllm", "openai", "av"):
        try:
            rows.append((mod, __import__(mod).__version__))
        except Exception:  # noqa: BLE001 — not installed is fine for packages this profile does not use
            rows.append((mod, "not installed"))
    display(pd.DataFrame(rows, columns=["setting", "value"]).set_index("setting"))

The next cell applies the settings and prints them, with the versions of the main packages and the
compute device. Check that `data_root` points at your data and that the device is `mps` on the Mac.

In [ ]:
REPO = Path(os.environ.get("NOTEBOOK_REPO", Path.cwd().parent)).resolve()
S = resolve_settings(dict(PROFILE=PROFILE, DRY_RUN=DRY_RUN, DATA_ROOT=DATA_ROOT, RUN=RUN, RUN_LIMIT=RUN_LIMIT,
                          RUN_STAGES=RUN_STAGES, REBUILD_POOL=REBUILD_POOL, REBUILD_SAMPLE=REBUILD_SAMPLE), REPO)
S["runs"].mkdir(parents=True, exist_ok=True)
show_settings(S)

<a id="s3"></a>
## 3. Helpers

**What.** A few small tools used by every later section:

- **`Store`** saves results as tables of JSON lines, one file per table (`items.jsonl`, `generations.jsonl`,
  …). Rows are only ever appended, each with a key; a key that is already stored is skipped. This is what
  makes every step safe to stop and re-run.
- **`short_id`** turns a list of strings into a 16-character ID (SHA-1). A generation's ID is built from the
  question, the system and its full configuration, so changing a model or a frame setting makes new rows
  instead of overwriting old ones.
- **`normalize_answer`** puts an answer in a standard form (lower case, no punctuation, no leading
  "a/an/the"), so that "A dog." and "dog" count as the same answer.
- **`need`** stops a section with a clear message when an earlier section has not been run.

**Why.** Long runs get interrupted. With these, no result is lost or duplicated.

**Reads → writes.** Nothing yet; this section only defines tools.

**Mac vs server:** nothing to change.

In [ ]:
class Store:
    """Append-only tables of JSON lines in one folder; one row per key (the first one wins)."""

    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self._keys: dict[str, set] = {}

    def path(self, table: str) -> Path:
        return self.root / f"{table}.jsonl"

    def _rows(self, table: str) -> list[dict]:
        p = self.path(table)
        if not p.exists():
            return []
        rows = []
        lines = p.read_text(encoding="utf-8").splitlines()
        for i, line in enumerate(lines):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                if i == len(lines) - 1:  # a crash while writing the last line: ignore that line
                    continue
                raise
        return rows

    def load(self, table: str) -> pd.DataFrame:
        """The table as a DataFrame (empty if it does not exist yet), keeping the first row per key."""
        rows, seen, out = self._rows(table), set(), []
        for r in rows:
            k = r.get("_key")
            if k in seen:
                continue
            seen.add(k)
            out.append({c: v for c, v in r.items() if c != "_key"})
        return pd.DataFrame(out)

    def keys(self, table: str) -> set:
        if table not in self._keys:
            self._keys[table] = {r.get("_key") for r in self._rows(table)}
        return self._keys[table]

    def append(self, table: str, rows: list[dict], key: str) -> int:
        """Append rows whose `key` column is new; return how many were written."""
        known = self.keys(table)
        new = [r for r in rows if str(r[key]) not in known]
        if not new:
            return 0
        with open(self.path(table), "a", encoding="utf-8") as f:
            for r in new:
                f.write(json.dumps({"_key": str(r[key]), **r}, ensure_ascii=False, default=_json_default) + "\n")
                known.add(str(r[key]))
            f.flush()
            os.fsync(f.fileno())
        return len(new)

    def rebuild(self, table: str) -> Path | None:
        """Rename the table out of the way (with a timestamp) so the section can rebuild it."""
        p = self.path(table)
        self._keys.pop(table, None)
        if not p.exists():
            return None
        old = p.with_name(f"{table}.{datetime.now():%Y%m%d-%H%M%S}.old.jsonl")
        p.rename(old)
        return old


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"cannot save {type(o).__name__}")


def short_id(*parts: str) -> str:
    """16-hex SHA-1 of the parts joined by '|'."""
    return hashlib.sha1("|".join(parts).encode("utf-8")).hexdigest()[:16]


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def canonical_json(obj) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False, default=_json_default)


_PUNCT = re.compile("[" + re.escape(string.punctuation) + "]")
_ARTICLE = re.compile(r"^(a|an|the) ")


def normalize_answer(text: str) -> str:
    """NFKC, lower case, punctuation → spaces, single spaces, one leading article removed.
    A non-empty answer never becomes empty."""
    s = unicodedata.normalize("NFKC", text).lower()
    out = " ".join(_PUNCT.sub(" ", s).split())
    out = _ARTICLE.sub("", out, count=1)
    if not out and text.strip():
        return " ".join(s.split())
    return out


def need(df: pd.DataFrame, section: int, what: str) -> pd.DataFrame:
    """Stop with a clear message when an earlier section's output is missing."""
    if df is None or len(df) == 0:
        raise RuntimeError(f"No {what} yet: run section {section} first (and sections 2–3 after a restart).")
    return df

Open the results store for this profile.

In [ ]:
store = Store(S["runs"])
print("results folder:", store.root)
print("tables so far:", sorted(p.stem for p in store.root.glob("*.jsonl")) or "none")

<a id="s4"></a>
## 4. Load the benchmarks

**What.** Read the question files of the three benchmarks and put every question into one table, one row
per question:

| Benchmark | Files read (in `<DATA_ROOT>/<folder>/annotations/`) | Question type (`qtype`) comes from |
|---|---|---|
| MSVD-QA (`msvd`) | `test_qa.json`, `youtube_mapping.txt` | the first word: what / who / how / when / where / other |
| ActivityNet-QA (`anetqa`) | `test_q.json` + `test_a.json`, joined on `question_id` | the answer file's type code (motion, spatial, temporal, yes/no, color, object, location, number, other) |
| NExT-QA open-ended (`nextoe`) | `OE/test-…parquet`, plus `MC/test-…parquet` for the five options | the `type` column (e.g. CW, TN, DC) |

Each question gets an ID `benchmark:split:question-id`. NExT questions also get their five
multiple-choice options, used later by the optional multiple-choice check on the server.

**Why.** All later steps work on this one table, so every benchmark looks the same from here on.

**Reads → writes.** Reads the question files. Writes nothing (the table is kept in memory as `questions`).

**Mac vs server:** nothing to change. If this cell says it cannot find a folder, fix `DATA_ROOT` in
section 2.

In [ ]:
BENCH_DIRS = {"msvd": "msvd", "anetqa": "anetqa", "nextoe": "nextqa"}   # benchmark → folder under DATA_ROOT
MSVD_QTYPES = ("what", "who", "how", "when", "where")
ANET_TYPES = ["motion", "spatial", "temporal", "yes_no", "color", "object", "location", "number", "other"]


def _item(benchmark, split, qid, video_id, question, refs, qtype, mc_options=None, mc_answer_idx=None) -> dict:
    return dict(item_id=f"{benchmark}:{split}:{qid}", benchmark=benchmark, split=split, video_id=video_id,
                question=question, answers_ref=refs, qtype=qtype, mc_options=mc_options,
                mc_answer_idx=mc_answer_idx)


def load_msvd(raw_dir: Path, split: str = "test") -> list[dict]:
    raw_dir = Path(raw_dir)
    mapping = {}
    for line in (raw_dir / "youtube_mapping.txt").read_text(encoding="utf-8").splitlines():
        if line.strip():
            clip, vid = line.split()
            mapping[vid] = clip                      # "vid12" → the YouTube clip name
    rows = json.loads((raw_dir / f"{split}_qa.json").read_text(encoding="utf-8"))
    out = []
    for r in rows:
        m = re.match(r"\s*([a-z]+)", r["question"].lower())
        qtype = m.group(1) if m and m.group(1) in MSVD_QTYPES else "other"
        out.append(_item("msvd", split, str(r["id"]), mapping[f"vid{r['video_id']}"], r["question"],
                         [r["answer"]], qtype))
    return out


def load_anetqa(raw_dir: Path, split: str = "test") -> list[dict]:
    raw_dir = Path(raw_dir)
    qs = json.loads((raw_dir / f"{split}_q.json").read_text(encoding="utf-8"))
    answers = {a["question_id"]: a for a in json.loads((raw_dir / f"{split}_a.json").read_text(encoding="utf-8"))}
    out, dropped = [], 0
    for q in qs:
        a = answers.get(q["question_id"])
        if a is None:
            dropped += 1
            continue
        try:
            qtype = ANET_TYPES[int(a["type"])]
        except (ValueError, IndexError, TypeError):
            qtype = str(a["type"])
        out.append(_item("anetqa", split, q["question_id"], f"v_{q['video_name']}", q["question"],
                         [a["answer"]], qtype))
    if dropped:
        print(f"anetqa: {dropped} questions without an answer were dropped")
    return out


def load_nextoe(raw_dir: Path, split: str = "test") -> list[dict]:
    raw_dir = Path(raw_dir)
    oe = pd.read_parquet(raw_dir / "OE" / f"{split}-00000-of-00001.parquet")
    mc_path = raw_dir / "MC" / f"{split}-00000-of-00001.parquet"
    mc = {}
    if mc_path.exists():
        for r in pd.read_parquet(mc_path).to_dict("records"):
            mc[(str(r["video"]), int(r["qid"]))] = r
    out = []
    for r in oe.to_dict("records"):
        video, qid = str(r["video"]), int(r["qid"])
        refs = [str(r["answer"])]
        extra = r.get("additional_ref_answer")
        if isinstance(extra, str) and extra.strip():
            refs.append(extra)                     # some NExT questions have a second valid answer
        m = mc.get((video, qid))
        out.append(_item("nextoe", split, f"{video}_{qid}", video, r["question"], refs, str(r["type"]),
                         [str(m[f"a{k}"]) for k in range(5)] if m else None, int(m["answer"]) if m else None))
    return out


LOADERS = {"msvd": load_msvd, "anetqa": load_anetqa, "nextoe": load_nextoe}


def load_benchmarks(data_root: Path, split: str = "test") -> pd.DataFrame:
    """Every question of the three benchmarks in one table."""
    data_root = Path(data_root)
    rows = []
    for bench, folder in BENCH_DIRS.items():
        raw = data_root / folder / "annotations"
        if not raw.exists():
            raise FileNotFoundError(f"{raw} not found. Is DATA_ROOT right (section 2)? It is {data_root}. "
                                    "See section 1 to download the data.")
        rows += LOADERS[bench](raw, split)
    df = pd.DataFrame(rows)
    dup = df["item_id"][df["item_id"].duplicated()]
    if len(dup):
        raise ValueError(f"duplicate question ids: {list(dup[:5])}")
    return df

Load the questions and look at them: counts per benchmark and question type, then two example questions
per benchmark.

In [ ]:
questions = load_benchmarks(S["data_root"])
print(f"{len(questions):,} questions loaded")
display(questions.groupby(["benchmark", "qtype"]).size().unstack(fill_value=0))
display(questions.groupby("benchmark").head(2)[["item_id", "video_id", "question", "answers_ref", "qtype"]])

<a id="s5"></a>
## 5. Check the videos

**What.** For every question, find its video file in `<DATA_ROOT>/<folder>/videos/` (searching
sub-folders too, by file name) and check that it really decodes: the first frame, and a frame near the
end. Questions whose video is missing or broken stay in the table but are marked `available = False`,
with the reason.

**Why.** A missing or broken video would otherwise silently drop out later and bias the sample.

**Reads → writes.** Reads the videos. Writes `pool.jsonl`: every question with `video_path`, `available`
and `problem`.

**Time.** About 10–30 minutes the first time (about 10,000 videos are opened once each). After that the
saved table is reused; set `REBUILD_POOL = True` in section 2 to check again, for example after more
videos were downloaded.

**Mac vs server:** nothing to change.

In [ ]:
VIDEO_EXTS = [".mp4", ".avi", ".mkv", ".webm"]


def index_videos(root: Path, exts=VIDEO_EXTS) -> dict[str, Path]:
    """File name (without extension) → path, for every video under `root` (first in sorted order wins)."""
    idx: dict[str, Path] = {}
    for p in sorted(Path(root).rglob("*")):
        if p.is_file() and p.suffix.lower() in exts:
            idx.setdefault(p.stem, p)
    return idx


def probe_decodable(path: Path) -> bool:
    """True if PyAV can decode the first frame and a frame near the end."""
    import av
    try:
        with av.open(str(path)) as c:
            stream = c.streams.video[0]
            next(c.decode(stream))
            if c.duration:
                c.seek(int(c.duration * 0.9))
                next(c.decode(stream))
        return True
    except Exception:  # noqa: BLE001 — any failure means "cannot decode"
        return False


def check_videos(questions: pd.DataFrame, data_root: Path, probe=probe_decodable) -> pd.DataFrame:
    """Add video_path, available and problem to each question (one benchmark folder at a time)."""
    out = []
    for bench, group in questions.groupby("benchmark", sort=False):
        idx = index_videos(Path(data_root) / BENCH_DIRS[bench] / "videos")
        probed: dict[Path, bool] = {}
        for r in tqdm(group.to_dict("records"), desc=f"{bench} videos"):
            v = r["video_id"]
            names = [v, v[2:]] if v.startswith("v_") else [v]   # ActivityNet files may lack the "v_" prefix
            path = next((idx[n] for n in names if n in idx), None)
            if path is None:
                out.append({**r, "video_path": "", "available": False, "problem": "missing"})
                continue
            if path not in probed:
                probed[path] = probe(path)
            ok = probed[path]
            out.append({**r, "video_path": str(path), "available": ok, "problem": "" if ok else "undecodable"})
    return pd.DataFrame(out)

Run the check (or reload its saved result), then look at the counts. If many videos are missing, look at
a few in the `missing` list and compare with the files in your `videos/` folders.

In [ ]:
if S["rebuild_pool"]:
    print("old pool kept as", store.rebuild("pool"))
pool = store.load("pool")
if pool.empty:
    if "questions" not in globals():
        questions = load_benchmarks(S["data_root"])
    pool = check_videos(questions, S["data_root"])
    store.append("pool", pool.to_dict("records"), key="item_id")
    pool = store.load("pool")
else:
    print("using the saved video check (set REBUILD_POOL = True in section 2 to redo it)")

summary = pool.groupby("benchmark").agg(questions=("item_id", "size"), available=("available", "sum"))
summary["missing"] = pool[pool.problem == "missing"].groupby("benchmark").size()
summary["undecodable"] = pool[pool.problem == "undecodable"].groupby("benchmark").size()
display(summary.fillna(0).astype(int))
for bench, n_ok in summary["available"].items():
    if n_ok < S["samples"]["main"]:
        print(f"WARNING: {bench} has only {n_ok} usable questions, fewer than the main sample ({S['samples']['main']})")
display(pool[~pool.available].groupby(["benchmark", "problem"]).head(3)[["item_id", "video_id", "problem"]])

<a id="s6"></a>
## 6. Draw the samples

**What.** Pick the questions the rest of the notebook works on, separately for each benchmark and only
among questions whose video decodes:

- **main** (Mac: 1,000 per benchmark) — Run 2;
- **annotated** (Mac: 100), a subset of *main* — Run 1, and the questions Claude labels;
- **vid** (Mac: 30), a subset of *annotated* — reserved for later labelling *with* the video.

Picks are **stratified by question type**: each type gets its fair share (largest-remainder rounding), and
every MSVD type gets at least 15 questions. A fixed random seed (42) makes the sample identical every
time. Finally the annotated questions are split 40/60 into **dev** and **test** (for later calibration
experiments), again by type.

**Why.** Nesting means Run 2 reuses everything Run 1 computed. Stratifying keeps rare question types from
disappearing in a small sample.

**Reads → writes.** Reads `pool.jsonl`. Writes `items.jsonl`: the questions of the *main* sample, with the
flags `in_main`, `in_annotated`, `in_vid_subset`, `vid_rank` and `human_split`.

**Draw the sample once.** Redrawing (`REBUILD_SAMPLE = True`) after answers exist means new questions and
wasted work.

**Mac vs server:** nothing to change; the sizes come from the profile (Mac 1,000 / 100 / 30, server
2,000 / 300 / 200).

In [ ]:
def allocate(counts: dict[str, int], n: int, min_per: int = 0) -> dict[str, int]:
    """Split n across strata: largest-remainder proportional shares, then raise strata below `min_per`
    (capped at their size) one at a time, taking from the largest current share."""
    total = sum(counts.values())
    n = min(n, total)
    if n == 0:
        return {k: 0 for k in counts}
    quotas = {k: n * c / total for k, c in counts.items()}
    alloc = {k: math.floor(q) for k, q in quotas.items()}
    for k in sorted(counts, key=lambda k: (-(quotas[k] - alloc[k]), k))[: n - sum(alloc.values())]:
        alloc[k] += 1
    for k in sorted(counts):
        target = min(min_per, counts[k])
        while alloc[k] < target:
            donor = max((d for d in alloc if d != k and alloc[d] > min(min_per, counts[d])),
                        key=lambda d: (alloc[d], d), default=None)
            if donor is None:
                break
            alloc[donor] -= 1
            alloc[k] += 1
    return alloc


def _strata(rows: list[dict]) -> dict[str, list[dict]]:
    groups: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        groups[r["qtype"]].append(r)
    return {k: sorted(v, key=lambda r: r["item_id"]) for k, v in sorted(groups.items())}


def stratified_sample(rows: list[dict], n: int, seed: int = 42, min_per: int = 0) -> list[dict]:
    """Stratify by question type; inside each type (sorted by id) draw with one seeded random generator."""
    strata = _strata(rows)
    alloc = allocate({k: len(v) for k, v in strata.items()}, n, min_per)
    rng = random.Random(seed)
    out = []
    for k, members in strata.items():
        out += rng.sample(members, alloc[k])
    return sorted(out, key=lambda r: r["item_id"])


def _vid_ranks(vid: list[dict], seed: int) -> dict[str, int]:
    """Order for the vid subset: round-robin over question types, so any prefix stays balanced."""
    rng = random.Random(seed)
    queues = []
    for _, members in _strata(vid).items():
        members = members[:]
        rng.shuffle(members)
        queues.append(members)
    order = []
    while any(queues):
        for q in queues:
            if q:
                order.append(q.pop(0)["item_id"])
    return {iid: r for r, iid in enumerate(order)}


def assign_nested(pool: pd.DataFrame, sizes: tuple[int, int, int], seed: int, msvd_min_per: int) -> pd.DataFrame:
    """Flag main ⊃ annotated ⊃ vid per benchmark; only available questions are ever picked."""
    n_main, n_ann, n_vid = sizes
    flags: dict[str, dict] = {}
    for bench in sorted(pool["benchmark"].unique()):
        rows = pool[(pool.benchmark == bench) & pool.available.astype(bool)].to_dict("records")
        min_per = msvd_min_per if bench == "msvd" else 0
        main = stratified_sample(rows, n_main, seed, min_per)
        ann = stratified_sample(main, n_ann, seed, min_per)
        vid = stratified_sample(ann, n_vid, seed)
        ranks = _vid_ranks(vid, seed)
        for r in main:
            flags.setdefault(r["item_id"], {})["in_main"] = True
        for r in ann:
            flags[r["item_id"]]["in_annotated"] = True
        for r in vid:
            flags[r["item_id"]].update(in_vid_subset=True, vid_rank=ranks[r["item_id"]])
    base = dict(in_main=False, in_annotated=False, in_vid_subset=False, vid_rank=None)
    return pd.DataFrame([{**r, **base, **flags.get(r["item_id"], {})} for r in pool.to_dict("records")])


def split_dev_test(df: pd.DataFrame, frac_dev: float = 0.4, seed: int = 42) -> pd.DataFrame:
    """human_split = dev / test for annotated questions (by type, per benchmark); None for the others."""
    dev = set()
    for bench in sorted(df["benchmark"].unique()):
        ann = df[(df.benchmark == bench) & df.in_annotated].to_dict("records")
        dev |= {r["item_id"] for r in stratified_sample(ann, round(frac_dev * len(ann)), seed)}
    df = df.copy()
    df["human_split"] = [("dev" if iid in dev else "test") if a else None
                         for iid, a in zip(df["item_id"], df["in_annotated"])]
    return df

Draw the samples (or reload them), and check the counts.

In [ ]:
if S["rebuild_sample"]:
    print("old sample kept as", store.rebuild("items"))
items = store.load("items")
if items.empty:
    pool = need(store.load("pool"), 5, "video check")
    sz = S["samples"]
    drawn = split_dev_test(assign_nested(pool, (sz["main"], sz["annotated"], sz["vid"]), S["seed"],
                                         S["msvd_min_per"]), S["frac_dev"], S["seed"])
    store.append("items", drawn[drawn.in_main].to_dict("records"), key="item_id")
    items = store.load("items")
else:
    print("using the saved sample (set REBUILD_SAMPLE = True in section 2 to redraw it)")

counts = items.groupby("benchmark").agg(main=("in_main", "sum"), annotated=("in_annotated", "sum"),
                                        vid=("in_vid_subset", "sum"))
counts["dev"] = items[items.human_split == "dev"].groupby("benchmark").size()
counts["test"] = items[items.human_split == "test"].groupby("benchmark").size()
display(counts.fillna(0).astype(int))
display(items[items.in_annotated].groupby(["benchmark", "qtype"]).size().unstack(fill_value=0))

<a id="s7"></a>
## 7. Frames

**What.** Video models do not watch the whole video; they get a handful of still frames. For a video
with `T` frames and `n` frames wanted, frame `i` is taken from the middle of the `i`-th of `n` equal
segments: index `int((i + 0.5) · T / n)`. Each frame is shrunk so its longest side is at most
`frame_max_side` pixels. The result is cached as a small `.npz` file, so every video is decoded only once,
however many systems use it.

**Why.** Evenly spaced frames are the standard, simple choice and they are the same for every model.
Caching saves a lot of time on long ActivityNet videos.

**Reads → writes.** Reads the video files. Writes `frames/` in the results folder.

**Mac vs server:** nothing to change here. The profile sets the numbers: Mac 8 frames at 448 px;
server 16 frames (ActivityNet 32) at 768 px. On a Mac that runs out of memory, lower `n_frames` or
`frame_max_side` in the profile cell of section 2.

In [ ]:
class FrameDecodeError(RuntimeError):
    pass


def sample_indices(n_total: int, n: int, start: int = 0, end: int | None = None) -> list[int]:
    """Centre of each of n equal segments of [start, end)."""
    length = (end if end is not None else n_total) - start
    return [start + int((i + 0.5) * length / n) for i in range(n)]


def _frame_count_and_fps(path: Path) -> tuple[int, float]:
    import av
    with av.open(str(path)) as c:
        s = c.streams.video[0]
        fps = float(s.average_rate or s.guessed_rate or 0) or 1.0
        n = s.frames
        if not n:  # some containers do not store the count: count by decoding
            n = sum(1 for _ in c.decode(s))
    return n, fps


def _decode_frames(path: Path, wanted: list[int], max_side: int) -> np.ndarray:
    import av
    from PIL import Image
    want, got = set(wanted), {}
    with av.open(str(path)) as c:
        for i, frame in enumerate(c.decode(c.streams.video[0])):
            if i in want:
                img = frame.to_image()
                img.thumbnail((max_side, max_side), Image.BICUBIC)
                got[i] = np.asarray(img.convert("RGB"), dtype=np.uint8)
            if i >= max(wanted):
                break
    missing = [i for i in wanted if i not in got]
    if missing:
        raise FrameDecodeError(f"frames {missing[:5]} not decodable in {path}")
    return np.stack([got[i] for i in wanted])


class FrameCache:
    """n evenly spaced frames of a video, resized to max_side, decoded once and cached on disk."""

    def __init__(self, root: Path, max_side: int = 768):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_side = max_side

    def get(self, video_path, video_id: str, n: int, t_start=None, t_end=None) -> tuple[np.ndarray, list[int]]:
        key = self.root / f"{video_id.replace('/', '_')}_{n}_centered_{t_start}_{t_end}_{self.max_side}.npz"
        if key.exists():
            z = np.load(key)
            return z["frames"], z["indices"].tolist()
        try:
            total, fps = _frame_count_and_fps(Path(video_path))
            start = int(t_start * fps) if t_start is not None else 0
            end = min(int(t_end * fps), total) if t_end is not None else total
            idx = sample_indices(total, n, start, end)
            arr = _decode_frames(Path(video_path), idx, self.max_side)
        except FrameDecodeError:
            raise
        except Exception as e:  # noqa: BLE001 — PyAV raises many error types; always name the video
            raise FrameDecodeError(f"{video_id}: cannot decode {video_path}: {e}") from e
        tmp = key.with_suffix(".tmp.npz")
        np.savez(tmp, frames=arr, indices=np.asarray(idx))
        tmp.replace(key)
        return arr, idx


def show_frames(frames: np.ndarray, title: str = "") -> None:
    """Show frames side by side (matplotlib if installed, otherwise one combined image)."""
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, len(frames), figsize=(2.2 * len(frames), 2.2))
        for ax, f in zip(np.atleast_1d(axes), frames):
            ax.imshow(f)
            ax.axis("off")
        fig.suptitle(title)
        plt.show()
    except ImportError:
        from PIL import Image
        display(Image.fromarray(np.concatenate(list(frames), axis=1)))

Look at the frames of one sampled video, as the video models will see them.

In [ ]:
items = need(store.load("items"), 6, "samples")
frame_cache = FrameCache(S["runs"] / "frames", S["frame_max_side"])
example = items[items.available.astype(bool)].iloc[0]
n_example = S["generators"][0]["n_frames"].get(example.benchmark, S["generators"][0]["n_frames"]["default"])
frames, idx = frame_cache.get(example.video_path, example.video_id, n_example)
print(f"{example.item_id}: frames {idx}, each {frames.shape[1]}×{frames.shape[2]} px")
print("Q:", example.question, "| reference:", example.answers_ref)
show_frames(frames, example.video_id)

<a id="s8"></a>
## 8. Generate answers

**What.** Every *system* answers every question of the current run (`RUN` in section 2). A system is a
video model plus a way of asking it:

| System name | How it is asked |
|---|---|
| `<model>` | with the frames, short answer ("Answer the question using a single word or phrase.") |
| `<model>-sentence` | with the frames, one full sentence plus a short reason |
| `<model>-blind` | **without** any frames: the model can only guess from the question |
| `<model>-f<k>` | with only k frames (server only: frame-count check) |

For each model, in turn:

1. load it once (on the Mac: Hugging Face `transformers` on the Apple GPU; on the server: vLLM);
2. for each question, build the chat message (frames placeholder + question + instruction), let the
   model's own processor turn it into the exact prompt, and generate greedily (no randomness);
3. save the answer, its normalised form, and the exact model version used;
4. free the model before loading the next one, so only one model is in memory.

Each answer's ID is a hash of the question, the system and its full set-up, so re-running skips
everything already answered and a changed set-up produces new answers rather than overwriting old ones.

**Why.** The *blind* and *sentence* variants are there on purpose: they create the "guessing without
looking" and "long answers" conditions whose judging the project studies.

**Reads → writes.** Reads `items.jsonl`, the frames, and the prompts in `../configs/prompts/generation/`.
Writes `generations.jsonl`.

**Time.** Mac, Run 1: 300 questions × 5 systems = 1,500 answers, about 1–2 hours. Run 2: about
13,500 more, 8–15 hours. Use `RUN_LIMIT = 5` (section 2) for a 5-minute first test.

**Mac vs server:** nothing to change in this section. The profile picks the engine: `transformers` on
the Mac, `vllm` on the server. If a model gives empty or garbled answers on the Mac, set its `dtype` to
`"bfloat16"` (or `"float32"`) in the profile cell.

In [ ]:
GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0)
OOM_HINT = "lower n_frames or frame_max_side in the profile cell of section 2"


def frames_for(gen: dict, benchmark: str) -> int:
    return gen["n_frames"].get(benchmark, gen["n_frames"]["default"])


def expand_systems(gen: dict, benchmark: str) -> list[dict]:
    """The systems of one model: one per style, plus blind and frame-ablation variants."""
    n = frames_for(gen, benchmark)
    out = [dict(system_id=gen["id"] if style == "short" else f"{gen['id']}-{style}", generator_id=gen["id"],
                style=style, blind=False, n_frames=n) for style in gen["styles"]]
    if gen["blind"]:
        out.append(dict(system_id=f"{gen['id']}-blind", generator_id=gen["id"], style="short", blind=True,
                        n_frames=0))
    out += [dict(system_id=f"{gen['id']}-f{k}", generator_id=gen["id"], style="short", blind=False, n_frames=k)
            for k in gen["frame_ablation"]]
    return out


def mc_system(gen: dict) -> dict:
    return dict(system_id=f"{gen['id']}-mc", generator_id=gen["id"], style="mc", blind=False,
                n_frames=frames_for(gen, "nextoe"))


def load_prompt(style: str, prompts_dir: Path) -> str:
    return (Path(prompts_dir) / "generation" / f"{style}.txt").read_text(encoding="utf-8").rstrip("\n")


def prompt_text(question: str, style: str, prompts_dir: Path, mc_options=None) -> str:
    """The instruction text with the question (and, for multiple choice, the options) filled in."""
    text = load_prompt(style, prompts_dir).replace("{question}", question)
    if style == "mc":
        for k, opt in enumerate(mc_options):
            text = text.replace("{a%d}" % k, opt)
    return text


def build_messages(family: str, text: str, blind: bool) -> list[dict]:
    """Chat messages for one question: a video placeholder (unless blind) and the text."""
    if family == "internvl":  # InternVL's template takes a plain string with a <video> placeholder
        return [{"role": "user", "content": text if blind else f"<video>\n{text}"}]
    content = ([] if blind else [{"type": "video"}]) + [{"type": "text", "text": text}]
    return [{"role": "user", "content": content}]


_LETTER = re.compile(r"\b([A-E])\b")


def parse_mc_letter(text: str) -> int | None:
    """Leading A–E not followed by a letter, else the first stand-alone A–E; A..E → 0..4."""
    s = text.strip()
    if s and s[0] in "ABCDE" and (len(s) == 1 or not s[1].isalpha()):
        return "ABCDE".index(s[0])
    m = _LETTER.search(s)
    return "ABCDE".index(m.group(1)) if m else None


class FakeEngine:
    """Stand-in model for dry runs: loads nothing, answers from a short fixed list."""
    batch_size = 1
    revision = "fake"

    def __init__(self, gen: dict):
        self.gen = gen

    def render(self, messages: list[dict]) -> str:
        return json.dumps(messages)

    def generate(self, requests: list[dict], max_new_tokens: int) -> list[str]:
        out = []
        for r in requests:
            if "Options:" in r["prompt"]:
                out.append("B")
                continue
            h = int(hashlib.sha1(r["prompt"].encode()).hexdigest(), 16)
            out.append(["a dog", "a man", "cooking", "I am not sure"][h % 4])
        return out

    def close(self) -> None:
        pass


class HFEngine:
    """A video model through Hugging Face transformers (Mac: Apple GPU). One request at a time."""
    batch_size = 1

    def __init__(self, gen: dict, device: str):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor
        self.gen, self.device, self.torch = gen, device, torch
        dtype = gen["dtype"]
        if dtype == "auto":
            dtype = {"mps": "float16", "cuda": "bfloat16"}.get(device, "float32")
        self.processor = AutoProcessor.from_pretrained(gen["repo"], revision=gen["revision"])
        self.model = AutoModelForImageTextToText.from_pretrained(gen["repo"], revision=gen["revision"],
                                                                 dtype=getattr(torch, dtype))
        self.model.to(device).eval()
        self.revision = getattr(self.model.config, "_commit_hash", None) or gen["revision"] or "latest"

    def render(self, messages: list[dict]) -> str:
        return self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    def generate(self, requests: list[dict], max_new_tokens: int) -> list[str]:
        out = []
        for r in requests:
            if r.get("video") is None:
                inputs = self.processor(text=[r["prompt"]], videos=None, return_tensors="pt")
            else:  # frames are already sampled: do not let the processor sample them again
                inputs = self.processor(text=[r["prompt"]], videos=[r["video"]], return_tensors="pt",
                                        do_sample_frames=False)
            inputs = inputs.to(self.device, dtype=self.model.dtype)
            with self.torch.no_grad():
                seq = self.model.generate(**inputs, max_new_tokens=max_new_tokens, **GREEDY)
            new = seq[0][inputs["input_ids"].shape[1]:]
            out.append(self.processor.batch_decode([new], skip_special_tokens=True)[0].strip())
        return out

    def close(self) -> None:
        self.model = self.processor = None
        gc.collect()
        if self.device == "mps":
            self.torch.mps.empty_cache()
        elif self.device == "cuda":
            self.torch.cuda.empty_cache()


class VLLMEngine:
    """A video model through vLLM on a CUDA GPU (server). Batches of 16."""
    batch_size = 16

    def __init__(self, gen: dict):
        from transformers import AutoProcessor, AutoTokenizer
        from vllm import LLM
        self.gen = gen
        if gen["family"] == "internvl":
            self.template = AutoTokenizer.from_pretrained(gen["repo"], revision=gen["revision"], trust_remote_code=True)
        else:
            self.template = AutoProcessor.from_pretrained(gen["repo"], revision=gen["revision"])
        self.llm = LLM(model=gen["repo"], revision=gen["revision"], dtype=gen["dtype"],
                       max_model_len=gen["max_model_len"], limit_mm_per_prompt={"video": 1}, seed=0,
                       trust_remote_code=gen["family"] == "internvl", enable_prefix_caching=False)
        self.revision = gen["revision"] or "latest"

    def render(self, messages: list[dict]) -> str:
        return self.template.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate(self, requests: list[dict], max_new_tokens: int) -> list[str]:
        from vllm import SamplingParams
        prompts = [{"prompt": r["prompt"], **({"multi_modal_data": {"video": r["video"]}} if r.get("video") is not None else {})}
                   for r in requests]
        outs = self.llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=max_new_tokens), use_tqdm=False)
        return [o.outputs[0].text.strip() for o in outs]

    def close(self) -> None:
        import torch
        del self.llm
        gc.collect()
        torch.cuda.empty_cache()


def make_engine(gen: dict, S: dict):
    if gen["backend"] == "fake":
        return FakeEngine(gen)
    if gen["backend"] == "vllm":
        return VLLMEngine(gen)
    return HFEngine(gen, detect_device(S["device"]).split(" ")[0])


def gen_config(gen: dict, system: dict, S: dict) -> dict:
    """Everything that determines an answer; its hash is part of the answer's ID."""
    return dict(model=gen["repo"], revision=gen["revision"], backend=gen["backend"], dtype=gen["dtype"],
                system=system, prompt=load_prompt(system["style"], S["prompts"]),
                frame_max_side=None if system["blind"] else S["frame_max_side"],
                max_new_tokens=S["max_new_tokens"][system["style"]], decoding="greedy")


def select_items(items: pd.DataFrame, S: dict, mc: bool = False) -> pd.DataFrame:
    """Questions of the current run that have a usable video (and options, for multiple choice)."""
    flag = "in_annotated" if S["run"] == "annotated" else "in_main"
    chosen = items[items[flag].astype(bool) & items["available"].astype(bool)].sort_values("item_id")
    if mc:
        chosen = chosen[(chosen.benchmark == "nextoe") & chosen["mc_options"].notna()]
    if S["run_limit"]:
        chosen = chosen.groupby("benchmark", sort=False).head(S["run_limit"])
    return chosen


def run_generation(store: Store, items: pd.DataFrame, S: dict, frame_cache=None, engine_factory=None,
                   mc: bool = False) -> int:
    """Answer every (question, system) pair of this run that is not stored yet; return how many were added."""
    need(items, 6, "samples")
    chosen = select_items(items, S, mc)
    frame_cache = frame_cache or FrameCache(S["runs"] / "frames", S["frame_max_side"])
    engine_factory = engine_factory or make_engine
    have, new, skipped = store.keys("generations"), 0, []
    for gen in S["generators"]:
        plan = []
        for bench in BENCHMARKS:
            rows = chosen[chosen.benchmark == bench].to_dict("records")
            for system in ([mc_system(gen)] if mc else expand_systems(gen, bench)):
                cfg = gen_config(gen, system, S)
                chash = sha256_text(canonical_json(cfg))[:16]
                todo = [(r, short_id(r["item_id"], system["system_id"], chash)) for r in rows]
                todo = [(r, gid) for r, gid in todo if gid not in have]
                if todo:
                    plan.append((system, chash, todo))
        if not plan:
            print(f"{gen['id']}: nothing to do")
            continue
        print(f"{gen['id']}: loading ({gen['backend']}) …")
        engine = engine_factory(gen, S)
        try:
            for system, chash, todo in plan:
                for b in tqdm(range(0, len(todo), engine.batch_size), desc=system["system_id"]):
                    reqs, kept = [], []
                    for r, gid in todo[b:b + engine.batch_size]:
                        frames, idx = None, []
                        if not system["blind"]:
                            try:
                                frames, idx = frame_cache.get(r["video_path"], r["video_id"], system["n_frames"])
                            except FrameDecodeError as e:
                                skipped.append((system["system_id"], r["item_id"], str(e)))
                                continue
                        text = prompt_text(r["question"], system["style"], S["prompts"], r.get("mc_options"))
                        msgs = build_messages(gen["family"], text, system["blind"])
                        reqs.append({"prompt": engine.render(msgs), "video": frames})
                        kept.append((r, gid, idx))
                    if not reqs:
                        continue
                    try:
                        answers = engine.generate(reqs, S["max_new_tokens"][system["style"]])
                    except Exception as e:  # noqa: BLE001
                        if "out of memory" in str(e).lower():
                            raise RuntimeError(f"{system['system_id']}: {e} — {OOM_HINT}") from e
                        raise
                    rows_out = [dict(gen_id=gid, item_id=r["item_id"], benchmark=r["benchmark"],
                                     system_id=system["system_id"], generator_id=gen["id"],
                                     generator_repo=gen["repo"], generator_rev=engine.revision,
                                     style=system["style"], blind=system["blind"], n_frames=system["n_frames"],
                                     frame_indices=list(idx), output_text=a, answer_norm=normalize_answer(a),
                                     config_hash=chash) for (r, gid, idx), a in zip(kept, answers)]
                    new += store.append("generations", rows_out, key="gen_id")
        finally:
            engine.close()
    for sid, iid, why in skipped[:20]:
        print(f"skipped {sid} {iid}: {why}")
    if skipped:
        print(f"{len(skipped)} answers skipped because a video could not be decoded")
    return new


def coverage(store: Store, items: pd.DataFrame, S: dict) -> pd.DataFrame:
    """How many questions of this run each system has answered, per benchmark."""
    chosen = select_items(items, S)
    gens = store.load("generations")
    rows = []
    for bench in BENCHMARKS:
        ids = set(chosen.loc[chosen.benchmark == bench, "item_id"])
        for gen in S["generators"]:
            for system in expand_systems(gen, bench):
                done = 0 if gens.empty else gens[(gens.system_id == system["system_id"]) & gens.item_id.isin(ids)].item_id.nunique()
                rows.append(dict(benchmark=bench, system=system["system_id"], questions=len(ids), answered=done,
                                 missing=len(ids) - done))
    return pd.DataFrame(rows)

The systems of this profile, per benchmark (number of frames each one sees):

In [ ]:
display(pd.DataFrame([dict(benchmark=b, **s) for g in S["generators"] for b in BENCHMARKS
                      for s in expand_systems(g, b)]).pivot(index="system_id", columns="benchmark", values="n_frames"))

Generate. The first time, each model is downloaded from the Hugging Face Hub (Mac: about 13 GB for the
three models). Progress bars show each system. You can stop this cell at any time and run it again later.

In [ ]:
items = need(store.load("items"), 6, "samples")
added = run_generation(store, items, S)
print(f"{added} new answers")
display(coverage(store, items, S).pivot(index="system", columns="benchmark", values="missing")
        .rename_axis(columns="missing answers per benchmark"))

A few answers side by side: the same questions, every system. The `generator_rev` column of
`generations.jsonl` records the exact model version used; copy it into `revision` in the profile cell to
pin a model.

In [ ]:
gens = need(store.load("generations"), 8, "answers")
sample_q = select_items(items, S).groupby("benchmark").head(2)
view = gens[gens.item_id.isin(sample_q.item_id)].pivot_table(index="item_id", columns="system_id",
                                                              values="output_text", aggfunc="first")
display(sample_q.set_index("item_id")[["question", "answers_ref"]].join(view))
display(gens.groupby("generator_id")["generator_rev"].first().rename("model version used"))

**Optional (server): multiple-choice check.** NExT-QA questions also have five options with a known
right answer. Asking the models to pick a letter gives an accuracy that needs **no judge at all**, a useful
anchor for the judged numbers. Set `RUN_MC_ANCHOR = True` in the next cell to run it (it adds one `-mc`
system per model, NExT questions only).

**Mac vs server:** optional on both; it was planned for the server.

In [ ]:
RUN_MC_ANCHOR = False
if RUN_MC_ANCHOR:
    run_generation(store, items, S, mc=True)
    gens = store.load("generations")
    mc = gens[gens["style"] == "mc"].merge(items[["item_id", "mc_answer_idx"]], on="item_id")
    mc["pred"] = mc["output_text"].map(parse_mc_letter)
    table = mc.groupby("system_id").agg(n=("item_id", "size"),
                                        accuracy=("pred", lambda s: (s == mc.loc[s.index, "mc_answer_idx"]).mean()),
                                        unparseable=("pred", lambda s: s.isna().sum()))
    display(table)

<a id="s9"></a>
## 9. Judge the answers

**What.** A judge model reads the question, the benchmark's reference answer and a system's answer, and
decides whether the answer is correct. Two prompt templates are used, read from
`../configs/prompts/judge/`:

| Template | Origin | The judge must reply with |
|---|---|---|
| **T-VCG** | the official Video-ChatGPT judge prompt, used by most papers | a dict like `{'pred': 'yes', 'score': 4}` |
| **T-RUB** | a yes/no rubric (direct match / indirect context / different context) | one word: `Yes` or `No` |

For every reply the notebook saves:

- the **verdict** (yes/no). A reply that does not follow the format is marked `unparseable` or
  `off_vocab` and is **never guessed**;
- the **margin** = log P(yes) − log P(no) at the token where the judge states its verdict, adding up all
  spellings (`Yes`, ` yes`, `'yes'`, …). +3 means about 95% yes, 0 is a coin flip. It lets later analyses
  move the yes/no threshold without asking the judge again;
- the top 5 alternative tokens and the raw reply.

Judging happens in **stages**, run in a fixed order; each stage is one judge × one template × one set:

| Mac stage | Set | Judge | Template | | Server stage | Set | Judge | Template |
|---|---|---|---|---|---|---|---|---|
| M1 | annotated | Qwen3-4B | T-VCG | | S1 | annotated | GPT-3.5 (0125) | T-VCG |
| M2 | annotated | Qwen3-4B | T-RUB | | S2 | annotated | GPT-3.5 (0125) | T-RUB |
| M3 | annotated | Qwen2.5-1.5B | T-VCG | | S3, S4 | main | GPT-3.5 (0125) | T-VCG, T-RUB |
| M4 | annotated | Qwen2.5-1.5B | T-RUB | | S5 | annotated | GPT-3.5 (0125) | T-VCG, 3 random samples |
| M5 | main | Qwen3-4B | T-VCG | | S6a–d | both | GPT-3.5 (0125) | frame-ablation systems |
| M6 | main | Qwen3-4B | T-RUB | | S7 | annotated | GPT-3.5 (1106) | T-VCG |
| G1, G2 *(optional)* | annotated | GPT-3.5 | T-VCG, T-RUB | | | | | |

By default this section runs the stages of the current `RUN`: M1–M4 for Run 1 and M5–M6 for Run 2 on
the Mac. A judge call that is already saved is skipped, so an answer in both sets is judged only once.

**Why.** Two judges of different size and two templates show how much the score depends on the judge
set-up rather than on the model being judged.

**Reads → writes.** Reads `items.jsonl`, `generations.jsonl` and the judge prompts. Writes
`judgecalls.jsonl`, and `judge_cache/` for API replies.

**Time.** Mac, Run 1: 6,000 judge calls, about 1–3 hours. Run 2: about 27,000, 8–20 hours.

**Mac vs server:** nothing to change here. On the Mac the local judges run with `transformers` on the
Apple GPU (one judge model in memory at a time). On the server, GPT-3.5 is called through the OpenAI API:
set `OPENAI_API_KEY` before starting Jupyter. Spending is capped per judge by `max_usd` in the profile.
To run the optional GPT-3.5 stages on the Mac, set `RUN_STAGES = ["G1", "G2"]` in section 2 (needs a key).

In [ ]:
JUDGE_FILES = {"t-vcg": ("t_vcg_system.txt", "t_vcg_user.txt"), "t-rub": ("t_rub.txt",)}
_SLOT = re.compile(r"\{(question|answer|pred)\}")


def load_template(template_id: str, prompts_dir: Path) -> tuple[tuple[str, ...], str]:
    """(template parts, sha256 of the template text). T-VCG has a system and a user part."""
    parts = tuple((Path(prompts_dir) / "judge" / f).read_text(encoding="utf-8") for f in JUDGE_FILES[template_id])
    digest = hashlib.sha256(b"\x00".join(p.encode() for p in parts)).hexdigest()
    return parts, digest


def protocol_id(template_id: str, decoding_tag: str, prompts_dir: Path) -> str:
    """'t-vcg@<first 8 hex of the template hash>/<decoding>': changes whenever the prompt text changes."""
    return f"{template_id}@{load_template(template_id, prompts_dir)[1][:8]}/{decoding_tag}"


def fill(text: str, question: str, answer: str, pred: str) -> str:
    """Fill {question}, {answer}, {pred} in one pass, so braces inside the values stay as they are."""
    values = {"question": question, "answer": answer, "pred": pred}
    return _SLOT.sub(lambda m: values[m.group(1)], text)


def render(template_id: str, question: str, answer: str, pred: str, prompts_dir: Path) -> list[dict]:
    parts, _ = load_template(template_id, prompts_dir)
    if template_id == "t-vcg":
        return [{"role": "system", "content": parts[0]}, {"role": "user", "content": fill(parts[1], question, answer, pred)}]
    return [{"role": "user", "content": fill(parts[0], question, answer, pred)}]


# --- reading the verdict -------------------------------------------------------------------------------
_FENCE = re.compile(r"```[a-zA-Z]*\n?|```")
_PRED = re.compile(r"""pred['"]?\s*:\s*['"]?(yes|no)\b""", re.I)
_SCORE = re.compile(r"""score['"]?\s*:\s*['"]?([0-9]+(?:\.[0-9]+)?)""", re.I)


def _verdict(v):
    s = str(v).strip().lower()
    return True if s == "yes" else False if s == "no" else None


def _score(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def parse_vcg(text: str):
    """(verdict, score, status) from a T-VCG reply: a Python dict string, else a 'pred: yes' pattern."""
    t = _FENCE.sub("", text)
    a, b = t.find("{"), t.rfind("}")
    if a != -1 and b > a:
        try:
            d = ast.literal_eval(t[a:b + 1])
            if isinstance(d, dict) and _verdict(d.get("pred")) is not None:
                return _verdict(d["pred"]), _score(d.get("score")), "ok"
        except (ValueError, SyntaxError):
            pass
    m = _PRED.search(t)
    if not m:
        return None, None, "unparseable"
    s = _SCORE.search(t)
    return m.group(1).lower() == "yes", float(s.group(1)) if s else None, "ok"


def parse_yes_no(text: str):
    """(verdict, status) from a T-RUB reply: exactly yes or no, ignoring case and end punctuation."""
    s = text.strip().strip(".!,;:\"'").lower()
    if s == "yes":
        return True, "ok"
    if s == "no":
        return False, "ok"
    return None, "off_vocab"


# --- the yes/no margin -----------------------------------------------------------------------------------
_STRIP = " \t\n'\":"
_PRED_KEY = re.compile(r"""pred['"]?\s*:\s*['"]?""")


def token_class(tok: str):
    s = tok.strip(_STRIP).lower()
    return s if s in ("yes", "no") else None


def _lse(xs):
    m = max(xs)
    return m + math.log(sum(math.exp(x - m) for x in xs))


def margin_from_top(top):
    yes = [lp for t, lp in top if token_class(t) == "yes"]
    no = [lp for t, lp in top if token_class(t) == "no"]
    return _lse(yes) - _lse(no) if yes and no else None


def vcg_verdict_index(tokens: list[str]):
    """Index of the token that holds the value of 'pred' in a T-VCG reply, or None."""
    starts, text = [], ""
    for t in tokens:
        starts.append(len(text))
        text += t
    m = _PRED_KEY.search(text)
    if not m:
        return None
    for i, (s, t) in enumerate(zip(starts, tokens)):
        if s + len(t) > m.end():
            return i if token_class(t) is not None else None
    return None


def margin_at(content: list[dict], template_id: str):
    """(margin, top-5 [token, logprob]) at the verdict position: 'pred' value (T-VCG) or first token (T-RUB)."""
    idx = vcg_verdict_index([c["token"] for c in content]) if template_id == "t-vcg" else (0 if content else None)
    if idx is None or idx >= len(content):
        return None, []
    top = [(t["token"], t["logprob"]) for t in content[idx].get("top_logprobs") or []]
    top5 = [[t, lp] for t, lp in sorted(top, key=lambda x: -x[1])[:5]]
    return margin_from_top(top), top5


def p_yes(margin: float) -> float:
    return 1.0 / (1.0 + math.exp(-margin))


# --- judges: every judge answers .complete(messages, request) the same way ------------------------------
class ModelRetired(Exception):
    pass


class BudgetExceeded(Exception):
    pass


class FakeJudge:
    """Stand-in judge for dry runs: answers yes/no from a hash, with made-up token probabilities."""

    def __init__(self, cfg: dict):
        self.cfg, self.revision = cfg, "fake"

    def complete(self, messages: list[dict], request: dict) -> dict:
        yes = int(hashlib.sha1(json.dumps(messages).encode()).hexdigest(), 16) % 3 != 0
        word = "yes" if yes else "no"
        top = [{"token": w, "logprob": lp} for w, lp in ((word, -0.2), ("no" if yes else "yes", -1.9))]
        if request["max_tokens"] > 1:   # T-VCG style
            toks = ["{'", "pred", "':", " '", word, "',", " 'score", "':", " 4" if yes else " 1", "}"]
        else:
            toks, top = [word.capitalize()], [{"token": t["token"].capitalize(), "logprob": t["logprob"]} for t in top]
        content = [{"token": t, "logprob": -0.1, "top_logprobs": top if t.strip(_STRIP).lower() == word else []}
                   for t in toks]
        return dict(choices=[dict(text="".join(toks), content=content)] * request.get("n", 1), model="fake",
                    cost=0.0, cached=False)

    def close(self):
        pass


def content_from_logits(gen_ids: list[int], step_logits: list, decode, k: int) -> list[dict]:
    """OpenAI-style token list from raw logits: each generated token and its top-k alternatives."""
    out = []
    for tid, logits in zip(gen_ids, step_logits):
        x = np.asarray(logits, dtype=np.float64)
        lp = x - x.max() - np.log(np.exp(x - x.max()).sum())
        top = np.argpartition(-lp, k - 1)[:k] if 0 < k < lp.size else np.arange(lp.size if k > 0 else 0)
        top = sorted(top.tolist(), key=lambda j: -lp[j])
        out.append({"token": decode(int(tid)), "logprob": float(lp[tid]),
                    "top_logprobs": [{"token": decode(j), "logprob": float(lp[j])} for j in top]})
    return out


class LocalJudge:
    """An open judge model through transformers (Mac). Greedy unless the stage asks for sampling."""

    def __init__(self, cfg: dict, device: str):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.cfg, self.device, self.torch = cfg, device, torch
        dtype = {"mps": "float16", "cuda": "bfloat16"}.get(device, "float32")
        self.tok = AutoTokenizer.from_pretrained(cfg["model"], revision=cfg["revision"])
        self.model = AutoModelForCausalLM.from_pretrained(cfg["model"], revision=cfg["revision"],
                                                          dtype=getattr(torch, dtype)).to(device).eval()
        self.revision = getattr(self.model.config, "_commit_hash", None) or cfg["revision"] or "latest"

    def complete(self, messages: list[dict], request: dict) -> dict:
        torch = self.torch
        prompt = self.tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        enc = self.tok(prompt, return_tensors="pt", add_special_tokens=False).to(self.device)
        n_prompt = enc["input_ids"].shape[1]
        pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id
        temperature = 0.0 if request.get("temperature") == 0 else (request.get("temperature") or 1.0)
        choices = []
        for k in range(request.get("n", 1)):
            kw = dict(max_new_tokens=request["max_tokens"], output_logits=True, return_dict_in_generate=True,
                      pad_token_id=pad, repetition_penalty=1.0)
            if temperature == 0:
                kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
            else:  # plain sampling, like the API default
                torch.manual_seed(k)
                kw.update(do_sample=True, temperature=temperature, top_p=1.0, top_k=0)
            with torch.no_grad():
                out = self.model.generate(**enc, **kw)
            ids = out.sequences[0][n_prompt:].tolist()
            logits = [step[0].float().cpu().numpy() for step in out.logits][:len(ids)]
            content = content_from_logits(ids, logits, lambda i: self.tok.decode([i]), request.get("top_logprobs", 0))
            choices.append(dict(text=self.tok.decode(ids, skip_special_tokens=True), content=content))
        return dict(choices=choices, model=self.cfg["model"], cost=0.0, cached=False)

    def close(self):
        self.model = self.tok = None
        gc.collect()
        if self.device == "mps":
            self.torch.mps.empty_cache()
        elif self.device == "cuda":
            self.torch.cuda.empty_cache()


class APIJudge:
    """An OpenAI chat model with token probabilities; replies are cached on disk, retried, and budgeted."""

    def __init__(self, cfg: dict, cache_dir: Path, create=None, sleep=time.sleep):
        self.cfg, self.sleep, self.revision = cfg, sleep, cfg["model"]
        self.cache = Path(cache_dir) / cfg["id"]
        self.cache.mkdir(parents=True, exist_ok=True)
        if create is None:
            from openai import OpenAI   # needs OPENAI_API_KEY in the environment
            create = OpenAI().chat.completions.create
        self.create = create
        self.spent = sum(json.loads(p.read_text())["cost"] for p in self.cache.glob("*.json"))

    def complete(self, messages: list[dict], request: dict) -> dict:
        req = dict(model=self.cfg["model"], messages=messages, max_tokens=request["max_tokens"],
                   n=request.get("n", 1), logprobs=True, top_logprobs=request.get("top_logprobs", 20))
        if request.get("temperature") is not None:
            req["temperature"] = request["temperature"]
        path = self.cache / f"{sha256_text(canonical_json(req))}.json"
        if path.exists():
            return {**json.loads(path.read_text())["result"], "cached": True}
        est_in = sum(len(m["content"]) // 3 + 4 for m in messages)
        est = (est_in * self.cfg["price_in_per_mtok"] + req["max_tokens"] * req["n"] * self.cfg["price_out_per_mtok"]) / 1e6
        if self.spent + est > self.cfg["max_usd"]:
            raise BudgetExceeded(f"{self.cfg['id']}: spent {self.spent:.4f} USD, cap {self.cfg['max_usd']}")
        for attempt in range(6):
            try:
                resp = self.create(**req)
                break
            except Exception as e:  # noqa: BLE001 — sort API errors by status
                status = getattr(e, "status_code", None)
                if status == 404 or getattr(e, "code", None) == "model_not_found":
                    raise ModelRetired(f"{self.cfg['model']}: {e}") from e
                retriable = status is None or status in (408, 409, 429) or status >= 500
                if not retriable or attempt == 5:
                    raise
                self.sleep(2.0 * 2 ** attempt * (1 + random.random() * 0.25))
        resp = resp.model_dump() if hasattr(resp, "model_dump") else resp
        usage = resp.get("usage") or {}
        cost = (usage.get("prompt_tokens", 0) * self.cfg["price_in_per_mtok"]
                + usage.get("completion_tokens", 0) * self.cfg["price_out_per_mtok"]) / 1e6
        self.spent += cost
        choices = [dict(text=(c.get("message") or {}).get("content") or "",
                        content=(c.get("logprobs") or {}).get("content") or []) for c in resp["choices"]]
        result = dict(choices=choices, model=resp.get("model"), fingerprint=resp.get("system_fingerprint"), cost=cost)
        path.write_text(json.dumps({"request": req, "result": result, "cost": cost}))
        return {**result, "cached": False}

    def close(self):
        pass


def make_judge(cfg: dict, S: dict):
    if cfg["backend"] == "fake":
        return FakeJudge(cfg)
    if cfg["backend"] == "openai":
        return APIJudge(cfg, S["runs"] / "judge_cache")
    return LocalJudge(cfg, detect_device(S["device"]).split(" ")[0])


def judge_call_id(gen_id: str, judge_id: str, pid: str, sample_idx: int) -> str:
    return short_id(gen_id, judge_id, pid, "none", "text", str(sample_idx))


def system_groups(S: dict) -> dict[str, set]:
    """'core' = every system except frame-ablation ones; 'ablation' = the -f<k> systems."""
    allsys = {s["system_id"] for g in S["generators"] for b in BENCHMARKS for s in expand_systems(g, b)}
    ablation = {f"{g['id']}-f{k}" for g in S["generators"] for k in g["frame_ablation"]}
    return {"core": allsys - ablation, "ablation": ablation}


def run_stages(store: Store, items: pd.DataFrame, S: dict, judge_factory=None) -> pd.DataFrame:
    """Run the selected judge stages in order; return a summary table (jobs, done, failed per stage)."""
    need(items, 6, "samples")
    gens = need(store.load("generations"), 8, "answers")
    judge_factory = judge_factory or make_judge
    by_item = items.set_index("item_id")
    groups, have = system_groups(S), store.keys("judgecalls")
    judges = {j["id"]: j for j in S["judges"]}
    current, summary = (None, None), []
    try:
        for st in S["stages"]:
            flag = "in_annotated" if st["subset"] == "annotated" else "in_main"
            ids = set(by_item.index[by_item[flag].astype(bool)])
            todo = gens[gens.item_id.isin(ids) & gens.system_id.isin(groups[st["systems"]]) & (gens["style"] != "mc")]
            if S["run_limit"] and not todo.empty:
                keep = set(todo.sort_values("item_id").groupby("benchmark")["item_id"].apply(
                    lambda s: list(dict.fromkeys(s))[: S["run_limit"]]).explode())
                todo = todo[todo.item_id.isin(keep)]
            pid = protocol_id(st["template"], st["decoding_tag"], S["prompts"])
            n = st["request"].get("n", 1)
            jobs = [g for g in todo.sort_values("gen_id").to_dict("records")
                    if not all(judge_call_id(g["gen_id"], st["judge_id"], pid, k) in have for k in range(n))]
            row = dict(stage=st["name"], judge=st["judge_id"], template=st["template"], jobs=len(jobs), done=0,
                       failed=0, errors=[])
            summary.append(row)
            if not jobs:
                continue
            if current[0] != st["judge_id"]:
                if current[1] is not None:
                    current[1].close()          # one local judge model in memory at a time
                print(f"{st['name']}: loading judge {st['judge_id']} …")
                current = (st["judge_id"], judge_factory(judges[st["judge_id"]], S))
            judge = current[1]
            for g in tqdm(jobs, desc=f"{st['name']} {st['judge_id']} {st['template']}"):
                it = by_item.loc[g["item_id"]]
                msgs = render(st["template"], it["question"], it["answers_ref"][0], g["output_text"], S["prompts"])
                t0 = time.time()
                try:
                    reply = judge.complete(msgs, st["request"])
                except (ModelRetired, BudgetExceeded) as e:
                    row["errors"].append(f"stopped: {e}")
                    print(f"{st['name']} stopped: {e}")
                    break
                except Exception as e:  # noqa: BLE001 — record and continue; re-running retries it
                    row["failed"] += 1
                    row["errors"].append(f"{g['gen_id']}: {type(e).__name__}: {e}")
                    continue
                recs = []
                for k, ch in enumerate(reply["choices"]):
                    if st["template"] == "t-vcg":
                        verdict, score, status = parse_vcg(ch["text"])
                    else:
                        (verdict, status), score = parse_yes_no(ch["text"]), None
                    margin, top5 = margin_at(ch["content"], st["template"])
                    if status == "ok" and margin is None:
                        status = "api_no_logprob"
                    recs.append(dict(judge_call_id=judge_call_id(g["gen_id"], st["judge_id"], pid, k),
                                     gen_id=g["gen_id"], item_id=g["item_id"], benchmark=g["benchmark"],
                                     system_id=g["system_id"], judge_id=st["judge_id"], judge_rev=judge.revision,
                                     protocol_id=pid, template=st["template"], decoding_tag=st["decoding_tag"],
                                     stage=st["name"], sample_idx=k, parsed_verdict=verdict, vcg_score=score,
                                     parse_status=status, margin=margin,
                                     p_yes=None if margin is None else p_yes(margin), top_tokens=top5,
                                     raw_text=ch["text"], latency_s=round(time.time() - t0, 3),
                                     cost_usd=reply.get("cost", 0.0) / max(len(reply["choices"]), 1),
                                     api_model=reply.get("model")))
                row["done"] += store.append("judgecalls", recs, key="judge_call_id")
    finally:
        if current[1] is not None:
            current[1].close()
    return pd.DataFrame(summary)

Run the judge stages of this run (see `S["stages"]` in section 2). Stop and re-run at any time.

In [ ]:
items = need(store.load("items"), 6, "samples")
judge_summary = run_stages(store, items, S)
display(judge_summary.drop(columns="errors"))
for _, r in judge_summary.iterrows():
    for e in r["errors"][:10]:
        print(f"{r['stage']}: {e}")

How well did each judge follow its format, and how often did it say yes? `ok` should be close to 1.0; a
judge that often ignores the format is worth knowing about in itself.

In [ ]:
calls = need(store.load("judgecalls"), 9, "judge verdicts")
display(calls.groupby(["stage", "judge_id", "template"]).agg(
    calls=("judge_call_id", "size"),
    ok=("parse_status", lambda s: (s == "ok").mean()),
    yes_rate=("parsed_verdict", lambda s: (s == True).mean()),  # noqa: E712
    mean_margin=("margin", "mean")).round(3))
display(calls.sample(min(5, len(calls)), random_state=0)[["system_id", "stage", "raw_text", "parsed_verdict",
                                                          "margin", "top_tokens"]])

<a id="s10"></a>
## 10. Label with Claude

**What.** The small judges need something better to be compared against. Here Claude labels the answers
of Run 1 by hand, through files:

1. **Units.** Identical answers to the same question (after normalising: "A dog." = "dog") from different
   systems become one *unit*, so each distinct answer is labelled once.
2. **Export.** Units are written into `batch_NN.md` files of about 100 units each; a question is never split
   across two batches. Each file is complete on its own: the task, six labelling rules, nine error tags,
   the exact reply format, and the questions with their reference answers and candidate answers.
   **Claude never sees which model wrote an answer, nor what the judges said.**
3. **You** give each file to Claude (a new conversation per batch: attach or paste it, send it) and save
   Claude's whole reply as a text file in the `replies/` folder, e.g. `replies/batch_01.txt`.
4. **Import.** The next cells read every reply file, find the CSV block in it, and check every row (known
   unit id, label `correct`/`incorrect`, error tags E1–E9, no conflicting duplicates). If anything is wrong,
   **nothing is saved** and every problem is listed with its file and row; fix the file and run the cell
   again. Re-importing is safe: labels already saved are skipped.

**The honesty rule.** Claude's labels are a *reference judge*: a strong model reading the same text as the
small judges, without the video. They are saved under the source name `claude-manual` and are never
called human labels. Labels from a person use `human:<initials>`.

**Why.** Comparing every judge with the same careful reference shows which judge set-ups are too lenient
or too strict, and whether that differs between systems.

**Reads → writes.** Reads `items.jsonl` and `generations.jsonl`. Writes `units.jsonl`,
`annotation/claude/batch_NN.md` and `manifest.json`, then `labels.jsonl` from your reply files.

**Mac vs server:** nothing to change. (On the server, labels from people use the CSV sheets of the
optional cell at the end of this section.)

In [ ]:
LABEL_SOURCE_RE = re.compile(r"^(claude-manual(-[A-Za-z0-9_.]+)?|human:[A-Za-z0-9_.-]+)$")
ERROR_TAGS = {f"E{k}" for k in range(1, 10)}
CLAUDE_HEADER = """# VideoQA answer labelling — batch {nn} of {total}

You are labelling answers to questions about short videos. You will NOT see the videos. For each
candidate answer, decide whether it is correct **given the question and the reference answer(s)**.

## Rules
1. Correct if it means the same as any reference answer, including synonyms and paraphrases.
2. More specific is fine ("puppy" for "dog"); more general is wrong ("animal" for "dog") unless the
   question asks for a category.
3. Partial answers to a multi-part reference are incorrect unless the missing part is not asked for.
4. Listing several alternatives ("a cat or a dog") is incorrect. Hedging around one answer: judge the answer.
5. Refusals, "cannot tell", and non-answers are incorrect.
6. Ignore extra details you cannot check, unless they contradict the reference.

## Error tags (for incorrect answers; optional for correct ones)
E1 synonym/paraphrase · E2 too general/too specific · E3 partial · E4 extra wrong detail ·
E5 shares words with the reference but means something else · E6 several alternatives ·
E7 refusal/non-answer · E8 reference looks wrong or ambiguous · E9 question ambiguous

## Output format
Reply with ONE csv code block and nothing else. Header exactly:
`unit_id,label,error_tags,note`
One row per unit below. label is `correct` or `incorrect`. error_tags separated by `;` (may be empty).
note is optional; quote it if it contains a comma.

## Items
"""


def build_units(items: pd.DataFrame, gens: pd.DataFrame) -> pd.DataFrame:
    """One unit per (question, normalised answer) among the given questions; multiple-choice answers skipped."""
    by_item = items.set_index("item_id")
    groups: dict[tuple, list[dict]] = defaultdict(list)
    for g in gens.to_dict("records"):
        if g["item_id"] in by_item.index and g.get("style") != "mc":
            groups[(g["item_id"], g["answer_norm"])].append(g)
    rows = []
    for (item_id, norm), members in groups.items():
        members = sorted(members, key=lambda g: g["gen_id"])
        rows.append(dict(unit_id=short_id(item_id, norm), item_id=item_id, benchmark=by_item.loc[item_id, "benchmark"],
                         answer_norm=norm, display_answer=members[0]["output_text"],
                         gen_ids=[g["gen_id"] for g in members], system_ids=sorted({g["system_id"] for g in members})))
    cols = ["unit_id", "item_id", "benchmark", "answer_norm", "display_answer", "gen_ids", "system_ids"]
    return pd.DataFrame(rows, columns=cols).sort_values(["item_id", "unit_id"]).reset_index(drop=True)


def plan_batches(units: pd.DataFrame, batch_size: int = 100) -> list[pd.DataFrame]:
    """Batches of about batch_size units; all units of one question stay in the same batch."""
    batches, current = [], []
    for _, q_units in units.groupby("item_id", sort=False):
        rows = q_units.to_dict("records")
        if current and len(current) + len(rows) > batch_size:
            batches.append(current)
            current = []
        current = current + rows
    if current:
        batches.append(current)
    return [pd.DataFrame(b) for b in batches]


def load_manifest(out_dir: Path) -> dict:
    p = Path(out_dir) / "manifest.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


def _one_line(text) -> str:
    return " ".join(str(text).split())


def _cell(text) -> str:
    s = _one_line(text).replace("|", "\\|")
    return s if s else "(empty answer)"


def _new_batches(units: pd.DataFrame, out_dir: Path, batch_size: int):
    manifest = load_manifest(out_dir)
    done = {u for ids in manifest.values() for u in ids}
    pending = units[~units.unit_id.isin(done)]
    named = [(f"batch_{len(manifest) + k + 1:02d}", b) for k, b in enumerate(plan_batches(pending, batch_size))]
    return named, manifest


def _write_manifest(out_dir: Path, manifest: dict, named) -> None:
    manifest = {**manifest, **{name: list(b["unit_id"]) for name, b in named}}
    (Path(out_dir) / "manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")


def export_claude(units: pd.DataFrame, items: pd.DataFrame, out_dir: Path, batch_size: int = 100) -> list[Path]:
    """Write batch_NN.md files for units not exported before, and update manifest.json."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    by_item = items.set_index("item_id")
    named, manifest = _new_batches(units, out_dir, batch_size)
    total = len(manifest) + len(named)
    paths = []
    for name, batch in named:
        lines = [CLAUDE_HEADER.format(nn=name.split("_")[1], total=f"{total:02d}")]
        for q, (item_id, q_units) in enumerate(batch.groupby("item_id", sort=False), start=1):
            it = by_item.loc[item_id]
            lines += [f"### Q{q}. {_one_line(it['question'])}",
                      f"Reference answer(s): {' | '.join(_one_line(a) for a in it['answers_ref'])}",
                      "| unit_id | candidate answer |", "|---|---|"]
            lines += [f"| {u} | {_cell(a)} |" for u, a in zip(q_units["unit_id"], q_units["display_answer"])]
            lines.append("")
        p = out_dir / f"{name}.md"
        p.write_text("\n".join(lines), encoding="utf-8")
        paths.append(p)
    if named:
        _write_manifest(out_dir, manifest, named)
    return paths


def export_human(units: pd.DataFrame, items: pd.DataFrame, out_dir: Path, batch_size: int = 100) -> list[Path]:
    """Write batch_NN.csv sheets (empty label columns) for people, and update manifest.json."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    by_item = items.set_index("item_id")
    named, manifest = _new_batches(units, out_dir, batch_size)
    paths = []
    for name, batch in named:
        buf = io.StringIO()
        w = csv.writer(buf, lineterminator="\n")
        w.writerow(["unit_id", "benchmark", "question", "reference_answers", "candidate_answer", "label",
                    "error_tags", "note"])
        for u in batch.to_dict("records"):
            it = by_item.loc[u["item_id"]]
            w.writerow([u["unit_id"], u["benchmark"], _one_line(it["question"]), " | ".join(it["answers_ref"]),
                        _one_line(u["display_answer"]), "", "", ""])
        p = out_dir / f"{name}.csv"
        p.write_text(buf.getvalue(), encoding="utf-8")
        paths.append(p)
    if named:
        _write_manifest(out_dir, manifest, named)
    return paths


class LabelImportError(Exception):
    def __init__(self, errors: list[str]):
        super().__init__("\n".join(errors))
        self.errors = errors


_CSV_FENCE = re.compile(r"```[^\n]*\n(.*?)```", re.S)


def validate_source(source: str) -> None:
    if not LABEL_SOURCE_RE.match(source):
        raise ValueError(f"source {source!r} must be 'claude-manual', 'claude-manual-<suffix>' or 'human:<initials>'")


def parse_label_csv(text: str) -> list[dict]:
    """Rows of the reply's CSV: inside the first ``` block that has one, else the CSV after the header line
    (stopping at the first following line with no comma, i.e. closing prose)."""
    text = text.replace("\r\n", "\n").replace("\r", "\n").lstrip("﻿")
    blocks = [b for b in _CSV_FENCE.findall(text) if "unit_id" in b]
    lines = (blocks[0] if blocks else text).split("\n")
    start = next((k for k, ln in enumerate(lines) if ln.strip().lstrip("﻿").startswith("unit_id")), None)
    if start is None:
        raise LabelImportError(["no header line starting with unit_id (expected: unit_id,label,error_tags,note)"])
    kept = [lines[start].strip()]
    for ln in lines[start + 1:]:
        if not blocks and ln.strip() and "," not in ln:
            break
        kept.append(ln.strip())
    reader = csv.DictReader(io.StringIO("\n".join(kept)), skipinitialspace=True, restkey="_extra")
    reader.fieldnames = [f.strip().lower() for f in reader.fieldnames or []]
    rows = []
    for n, r in enumerate(reader, start=1):
        if not any((v or "").strip() for k, v in r.items() if k != "_extra" and isinstance(v, str)):
            continue
        tags = [t.strip().upper() for t in (r.get("error_tags") or "").split(";") if t.strip()]
        row = {"row": n, "unit_id": (r.get("unit_id") or "").strip(), "label": (r.get("label") or "").strip().lower(),
               "error_tags": tags, "note": (r.get("note") or "").strip()}
        if r.get("_extra"):
            row["too_many_fields"] = True
        rows.append(row)
    return rows


def _validate(rows: list[dict], batch_of: dict) -> list[str]:
    errors, seen = [], {}
    for r in rows:
        at = r["row"] if isinstance(r["row"], str) else f"row {r['row']}"
        where = f"{at} ({r['unit_id'] or 'no unit_id'})"
        if r.get("too_many_fields"):
            errors.append(f"{where}: more than 4 fields (quote a note that contains commas; separate tags with ;)")
        if r["unit_id"] not in batch_of:
            errors.append(f"{where}: unit_id {r['unit_id']!r} was not exported")
        if r["label"] not in ("correct", "incorrect"):
            errors.append(f"{where}: label {r['label']!r} must be correct or incorrect")
        bad = [t for t in r["error_tags"] if t not in ERROR_TAGS]
        if bad:
            errors.append(f"{where}: error tag(s) {', '.join(bad)} not in E1–E9")
        prev = seen.get(r["unit_id"])
        if prev is not None and prev != r["label"]:
            errors.append(f"{where}: unit {r['unit_id']} appears twice with different labels")
        seen.setdefault(r["unit_id"], r["label"])
    return errors


def import_labels(store: Store, rows: list[dict], source: str, manifest: dict, task: str = "ref") -> dict:
    """Check every row, then save the labels; labels already saved for (unit, source) are skipped."""
    validate_source(source)
    batch_of = {u: b for b, ids in manifest.items() for u in ids}
    errors = _validate(rows, batch_of)
    if errors:
        raise LabelImportError(errors)
    stored = skipped = 0
    for r in rows:
        rec = dict(label_id=short_id(r["unit_id"], source), unit_id=r["unit_id"], source=source, task=task,
                   label=r["label"], error_tags=r["error_tags"], note=r["note"], batch=batch_of[r["unit_id"]])
        if store.append("labels", [rec], key="label_id"):
            stored += 1
        else:
            skipped += 1
    labels = store.load("labels")
    labelled = set() if labels.empty else set(labels.loc[labels.source == source, "unit_id"])
    missing = {b: sum(u not in labelled for u in ids) for b, ids in manifest.items()}
    return dict(stored=stored, skipped_existing=skipped, missing_by_batch={b: k for b, k in missing.items() if k})

**Step 1 — export.** Builds the units for Run 1's questions (the *annotated* set) and writes the batch files.
Running it again later only adds batches for new answers; batches already labelled stay valid.

In [ ]:
ANNOTATION = S["runs"] / "annotation" / "claude"
(ANNOTATION / "replies").mkdir(parents=True, exist_ok=True)
items = need(store.load("items"), 6, "samples")
gens = need(store.load("generations"), 8, "answers")
annotated = items[items.in_annotated.astype(bool)]
units = build_units(annotated, gens)
store.append("units", units.to_dict("records"), key="unit_id")
new_files = export_claude(units, annotated, ANNOTATION)
print(f"{len(units)} units for {annotated.item_id.nunique()} questions; {len(new_files)} new batch file(s)")
print("batch files are in:", ANNOTATION)
for p in sorted(ANNOTATION.glob("batch_*.md")):
    print("  ", p.name)

**Step 2 — Claude.** For each `batch_NN.md`: open a new Claude conversation, attach or paste the file, send
it, and save Claude's **whole reply** as `replies/batch_NN.txt` in the folder printed above. Extra sentences
around the CSV block do no harm.

If a reply stops early, delete its last row if it is cut off mid-line, then ask Claude in the same
conversation: "Continue from the row after `<last complete unit_id>`, in a new csv block with the same
header." Save that as a second file (e.g. `batch_01b.txt`).

**Step 3 — import.** Reads every file in `replies/`. Set `LABEL_SOURCE = "claude-manual-v2"` to import a
second, corrected pass under a new name instead of changing the first one.

In [ ]:
LABEL_SOURCE = "claude-manual"
manifest = load_manifest(ANNOTATION)
reply_files = sorted((ANNOTATION / "replies").glob("*.txt")) + sorted((ANNOTATION / "replies").glob("*.csv"))
if not reply_files:
    print("No replies yet: save Claude's replies in", ANNOTATION / "replies", "and run this cell again.")
else:
    rows, problems = [], []
    for f in reply_files:
        try:
            rows += [{**r, "row": f"{f.name} row {r['row']}"} for r in parse_label_csv(f.read_text(encoding="utf-8"))]
        except LabelImportError as e:
            problems += [f"{f.name}: {m}" for m in e.errors]
    try:
        if problems:
            raise LabelImportError(problems)
        result = import_labels(store, rows, LABEL_SOURCE, manifest)
        print(f"saved {result['stored']} label(s) from {len(reply_files)} file(s); "
              f"{result['skipped_existing']} were already saved")
    except LabelImportError as e:
        for m in e.errors:
            print("PROBLEM:", m)
        print(f"{len(e.errors)} problem(s): nothing was saved. Fix the files above and run this cell again.")

**Step 4 — status.** Units per batch, labels per source, and units with no label yet.

In [ ]:
labels = store.load("labels")
status = []
for batch, ids in load_manifest(ANNOTATION).items():
    got = labels[labels.unit_id.isin(ids)] if not labels.empty else labels
    per_source = "" if got.empty else ", ".join(f"{s}: {k}" for s, k in got.groupby("source").size().items())
    status.append(dict(batch=batch, units=len(ids), labelled=per_source or "—",
                       missing=len(set(ids) - (set(got.unit_id) if not got.empty else set()))))
display(pd.DataFrame(status))

**Optional: sheets for people.** Writes `batch_NN.csv` files with empty `label` columns to
`annotation/human/`. Import a filled sheet with the import cell above after setting `LABEL_SOURCE` to
`"human:<initials>"` and pointing `ANNOTATION` to the `human` folder.

In [ ]:
EXPORT_HUMAN_SHEETS = False
if EXPORT_HUMAN_SHEETS:
    files = export_human(units, annotated, S["runs"] / "annotation" / "human")
    print(f"{len(files)} sheet(s) written to", S["runs"] / "annotation" / "human")

<a id="s11"></a>
## 11. Report

**What.** Put everything together, per benchmark and system:

- **accuracy under each judge set-up** (`judge/template/decoding`, e.g. `qwen3-4b/t-rub/t0`): the share of
  answers the judge called correct. `(k)` counts replies that could not be read; they are left out, never
  guessed. `[k of n judged]` appears when a judge has graded only part of the answers;
- **accuracy under each label source** (e.g. `claude-manual`);
- **agreement with the first label source**, using only answers that have both a verdict and a label:

  | Measure | Meaning | Read it as |
  |---|---|---|
  | gap (pts) | judge accuracy − reference accuracy, in percentage points | +6.0 = the judge is 6 points more generous |
  | κ (Cohen's kappa) | agreement corrected for chance (1 perfect, 0 chance) | > 0.8 very good, 0.6–0.8 good, < 0.4 weak |
  | FPR | of the answers the reference calls *incorrect*, the share the judge accepted | high = lenient judge |
  | FNR | of the answers the reference calls *correct*, the share the judge rejected | high = strict judge |

- **visual gain**: accuracy *with* the video minus accuracy of the same model *without* it, per judge
  set-up and per label source. If the judges' gain differs from the reference's gain, the judge distorts
  how much seeing the video helps.

**What to look for.** Compare the gap and the FPR *between systems* for the same judge. A judge that is
+2 points too generous for one system and +10 for another is unfair between them, and no single correction
can fix that. With 100 answers, an accuracy has an uncertainty of about ±10 points, so treat small
differences in Run 1 as noise; Run 2 (1,000 answers) narrows it to about ±3.

**Reads → writes.** Reads `items`, `generations`, `judgecalls` and `labels`. Writes `results.md`.

**Mac vs server:** nothing to change.

In [ ]:
REPORT_NOTE = "Label sources other than `human:*` are reference judges, not human ground truth."
DASH = "—"


def yes_rate(xs):
    return sum(xs) / len(xs) if xs else None


def cohen_kappa(a, b):
    """(p_o − p_e) / (1 − p_e); None when there are no pairs or p_e = 1."""
    if not a or len(a) != len(b):
        return None
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = sum(a) / n, sum(b) / n
    pe = pa * pb + (1 - pa) * (1 - pb)
    return None if pe == 1 else (po - pe) / (1 - pe)


def error_rates(judge, ref) -> dict:
    pos = [j for j, r in zip(judge, ref) if r]
    neg = [j for j, r in zip(judge, ref) if not r]
    return {"fpr": sum(neg) / len(neg) if neg else None, "fnr": sum(not j for j in pos) / len(pos) if pos else None,
            "n_ref_pos": len(pos), "n_ref_neg": len(neg)}


def _config_key(call: dict) -> str:
    return f"{call['judge_id']}/{call['protocol_id'].split('@')[0]}/{call['protocol_id'].split('/')[-1]}"


def _current_calls(store: Store, prompts_dir: Path) -> tuple[list[dict], int]:
    """First-sample judge calls made with the current prompt files, and how many older ones were left out."""
    current = {t: f"{t}@{load_template(t, prompts_dir)[1][:8]}" for t in JUDGE_FILES}
    calls = store.load("judgecalls")
    if calls.empty:
        return [], 0
    calls = [c for c in calls.to_dict("records") if c.get("sample_idx", 0) == 0]
    keep = [c for c in calls if current.get(c["protocol_id"].split("@")[0]) == c["protocol_id"].split("/")[0]]
    return keep, len(calls) - len(keep)


def build_tables(store: Store, items: pd.DataFrame, subset: str, prompts_dir: Path) -> dict:
    """{benchmark: {system: {n, configs, sources, agreement}}} for the answers of one set."""
    flag = "in_annotated" if subset == "annotated" else "in_main"
    chosen = items[items[flag].astype(bool) & items["available"].astype(bool)].set_index("item_id")
    calls, _ = _current_calls(store, prompts_dir)
    verdict: dict = defaultdict(dict)
    for c in calls:
        verdict[c["gen_id"]][_config_key(c)] = None if pd.isna(c["parsed_verdict"]) else bool(c["parsed_verdict"])
    labels: dict = defaultdict(dict)
    lab = store.load("labels")
    for r in ([] if lab.empty else lab.to_dict("records")):
        if r["task"] == "ref":
            labels[r["unit_id"]][r["source"]] = r["label"] == "correct"
    configs = sorted({_config_key(c) for c in calls})
    sources = sorted({s for per in labels.values() for s in per})
    gens = store.load("generations")
    groups: dict = defaultdict(lambda: defaultdict(list))
    for g in ([] if gens.empty else gens.to_dict("records")):
        if g["item_id"] in chosen.index and g.get("style") != "mc":
            groups[chosen.loc[g["item_id"], "benchmark"]][g["system_id"]].append(g)
    tables = {}
    for bench in sorted(groups):
        tables[bench] = {}
        for system in sorted(groups[bench]):
            rows = groups[bench][system]
            unit = {g["gen_id"]: short_id(g["item_id"], g["answer_norm"]) for g in rows}
            row = {"n": len(rows), "configs": {}, "sources": {}, "agreement": {}}
            for cfg in configs:
                vs = [verdict[g["gen_id"]][cfg] for g in rows if cfg in verdict[g["gen_id"]]]
                if vs:
                    parsed = [v for v in vs if v is not None]
                    row["configs"][cfg] = {"acc": yes_rate(parsed), "unparseable": len(vs) - len(parsed), "judged": len(vs)}
            for src in sources:
                ls = [labels[unit[g["gen_id"]]][src] for g in rows if src in labels.get(unit[g["gen_id"]], {})]
                if ls:
                    row["sources"][src] = {"acc": yes_rate(ls), "n": len(ls)}
            if sources:
                ref = sources[0]
                for cfg in configs:
                    pairs = [(verdict[g["gen_id"]][cfg], labels[unit[g["gen_id"]]][ref]) for g in rows
                             if verdict[g["gen_id"]].get(cfg) is not None and ref in labels.get(unit[g["gen_id"]], {})]
                    if pairs:
                        j, r = [p[0] for p in pairs], [p[1] for p in pairs]
                        rates = error_rates(j, r)
                        row["agreement"][cfg] = {"source": ref, "n": len(pairs),
                                                 "gap_pts": 100 * (yes_rate(j) - yes_rate(r)),
                                                 "kappa": cohen_kappa(j, r), "fpr": rates["fpr"], "fnr": rates["fnr"]}
            tables[bench][system] = row
    return tables


def visual_gain(tables: dict) -> dict:
    """Accuracy with video − accuracy of the same model blind, per judge set-up and label source."""
    out = {}
    for bench, rows in tables.items():
        for system, row in rows.items():
            blind = rows.get(f"{system}-blind")
            if blind is None:
                continue
            d = lambda a, b: None if a is None or b is None else a - b  # noqa: E731
            out.setdefault(bench, {})[system] = {
                "configs": {c: d(v["acc"], blind["configs"][c]["acc"]) for c, v in row["configs"].items() if c in blind["configs"]},
                "sources": {s: d(v["acc"], blind["sources"][s]["acc"]) for s, v in row["sources"].items() if s in blind["sources"]}}
    return out


def _f(x, fmt=".3f"):
    return DASH if x is None else format(x, fmt)


def accuracy_frame(rows: dict, with_sources: bool) -> pd.DataFrame:
    """One benchmark's accuracy table as a DataFrame (cells formatted like the markdown report)."""
    configs = sorted({k for r in rows.values() for k in r["configs"]})
    sources = sorted({k for r in rows.values() for k in r["sources"]}) if with_sources else []
    out = []
    for system, r in rows.items():
        rec = {"system": system, "n": r["n"]}
        for c in configs:
            v = r["configs"].get(c)
            rec[c] = DASH if v is None else (f"{_f(v['acc'])} ({v['unparseable']})" +
                                            ("" if v["judged"] >= r["n"] else f" [{v['judged']} of {r['n']} judged]"))
        for s in sources:
            rec[s] = _f(r["sources"][s]["acc"]) if s in r["sources"] else DASH
        out.append(rec)
    return pd.DataFrame(out)


def _md(df: pd.DataFrame) -> list[str]:
    cols = list(df.columns)
    return ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] + \
           ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)]


def write_results(store: Store, items: pd.DataFrame, profile: str, prompts_dir: Path, out: Path) -> Path:
    """Write the markdown report: Run 1 accuracy + agreement, visual gain, Run 2 accuracy."""
    run1 = build_tables(store, items, "annotated", prompts_dir)
    _, stale = _current_calls(store, prompts_dir)
    lines = [f"# Results — profile `{profile}` (notebook)", "", f"> {REPORT_NOTE}",
             "> acc = share of answers called correct. (k) = replies that could not be read; counted, never guessed. "
             "[k of n judged] = that judge saw only part of the answers.", ""]
    if stale:
        lines += [f"> {stale} judge call(s) made with an older version of a judge prompt are not counted.", ""]
    try:
        import yaml
        src = yaml.safe_load((Path(prompts_dir) / "judge" / "source.yaml").read_text(encoding="utf-8"))
        unverified = [t for t, v in src.items() if not v.get("verified")]
        if unverified:
            lines += [f"> Judge prompts not verified against their source: {', '.join(unverified)} "
                      "(see section 1c).", ""]
    except Exception:  # noqa: BLE001 — the note is informational
        pass
    lines += ["## Run 1 — annotated set", ""]
    if not run1:
        lines += ["No answers for the annotated set yet.", ""]
    for bench, rows in run1.items():
        lines += [f"### {bench}", ""] + _md(accuracy_frame(rows, True)) + [""]
        agree = [(s, c, a) for s, r in rows.items() for c, a in sorted(r["agreement"].items())]
        if not agree:
            lines += ["Agreement: no labels imported yet (section 10).", ""]
            continue
        lines += [f"Agreement with `{agree[0][2]['source']}` (gap = judge acc − reference acc, in points):", "",
                  "| system | judge config | n | gap (pts) | κ | FPR | FNR |", "|---|---|---|---|---|---|---|"]
        lines += [f"| {s} | {c} | {a['n']} | {_f(a['gap_pts'], '+.1f')} | {_f(a['kappa'])} | {_f(a['fpr'])} | {_f(a['fnr'])} |"
                  for s, c, a in agree] + [""]
    gain = visual_gain(run1)
    lines += ["## Visual gain (annotated set)", "", "Δ = accuracy with video − accuracy of the same model blind.", ""]
    if not gain:
        lines += ["No system has a blind partner with results yet.", ""]
    for bench, per in gain.items():
        for system, d in per.items():
            parts = [f"{c}: {_f(v, '+.3f')}" for c, v in sorted(d["configs"].items())]
            parts += [f"{s}: {_f(v, '+.3f')}" for s, v in sorted(d["sources"].items())]
            lines.append(f"- {bench} / {system}: " + ("; ".join(parts) or DASH))
    lines += ["", "## Run 2 — main set", ""]
    run2 = build_tables(store, items, "main", prompts_dir)
    if not run2:
        lines += ["No answers for the main set yet.", ""]
    for bench, rows in run2.items():
        lines += [f"### {bench}", ""] + _md(accuracy_frame(rows, False)) + [""]
    out = Path(out)
    out.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return out

Build the report, show the Run 1 tables here, and save everything to `results.md`.

In [ ]:
items = need(store.load("items"), 6, "samples")
need(store.load("judgecalls"), 9, "judge verdicts")
results_path = write_results(store, items, S["profile"], S["prompts"], S["runs"] / "results.md")
tables = build_tables(store, items, "annotated", S["prompts"])
for bench, rows in tables.items():
    print(f"\n{bench} — accuracy (annotated set)")
    display(accuracy_frame(rows, True).set_index("system"))
    agree = pd.DataFrame([dict(system=s, config=c, **{k: v for k, v in a.items() if k != "source"})
                          for s, r in rows.items() for c, a in r["agreement"].items()])
    if not agree.empty:
        source = next(a["source"] for r in rows.values() for a in r["agreement"].values())
        print(f"{bench} — agreement with {source}")
        display(agree.round(3).set_index(["system", "config"]))
gain = pd.DataFrame([dict(benchmark=b, system=s, setup=k, visual_gain=v) for b, per in visual_gain(tables).items()
                     for s, d in per.items() for k, v in {**d["configs"], **d["sources"]}.items()])
if not gain.empty:
    print("\nvisual gain (accuracy with video − blind)")
    display(gain.pivot_table(index=["benchmark", "system"], columns="setup", values="visual_gain").round(3))
print("\nreport written to", results_path)

<a id="s12"></a>
## 12. Troubleshooting and next steps

| Symptom | Cause and fix |
|---|---|
| Section 4: `… annotations not found` | `DATA_ROOT` is wrong. On the Mac the data is in `vjaudit/data` next to this repository; check the path printed in section 2. |
| `NameError: name 'S' is not defined` (or `store`, `items`) | The kernel restarted. Run sections 2 and 3, then the section you want. |
| `No … yet: run section N first` | That section's input does not exist yet; run section N. |
| Device shows `cpu` in section 2 | torch cannot see the Apple GPU. In a terminal: `python -c "import torch; print(torch.backends.mps.is_available())"` must print `True`; if not, `pip install --upgrade --force-reinstall torch torchvision`. |
| `out of memory` in section 8 or 9 | Lower `n_frames` (e.g. 6) or `frame_max_side` (e.g. 384) in the profile cell (section 2). Changed settings make new answers; old ones stay. |
| Empty or garbled answers from one model | Half precision can be unstable on the Apple GPU: set that model's `dtype` to `"bfloat16"` or `"float32"` in the profile cell. |
| A model fails to load in section 8 | The download failed or the name is wrong. Check the internet connection and the `repo` name. To skip a model, delete its entry from the profile cell. |
| Judge stage shows `failed` | The reasons are printed under the summary. Re-running section 9 retries only the failed calls. |
| `stopped: … BudgetExceeded` | An API judge reached its `max_usd` cap. Raise it in the profile cell on purpose, then re-run. |
| `stopped: … ModelRetired` | The OpenAI model no longer exists (GPT-3.5 is retired after 2026-10-23). |
| Section 10 lists problems | Nothing was saved. Fix the rows named in the messages (file and row number) and run the import cell again. |
| Model downloads fill the wrong disk | Set `HF_HOME=/path/with/space` before starting Jupyter. |

**Next steps.**

1. Run 1 done and labelled? Look at `results.md` (section 11).
2. Set `RUN = "main"` in section 2, then run sections 8, 9 and 11 again for Run 2. Run 1's answers and
   verdicts are reused automatically.
3. The command-line version of the same pipeline (`make …`, see `../WORKFLOW.md`) keeps its own results in
   `artifacts/`; the two do not share results.